# 구강질환 이미지 분류 프로젝트 - GitHub 업로드용 정리본

이 노트북은 **프로젝트 보고서 흐름**에 맞춰 정리한 실행본입니다.  
본문에는 최종 분석 경로만 남기고, 이전 시행착오·주석 처리 실험·중복 블록은 맨 뒤 **부록**으로 분리했습니다.

메인 흐름은 아래 순서로 구성했습니다.

1. 데이터 수집 / 데이터셋 구축 / EDA  
2. 학습 파이프라인 구축 및 안정화  
3. 원본 데이터셋 기준 모델 학습 결과  
4. 데이터 재분할 원인 분석 및 sharpness 기반 재분할  
5. 재분할 데이터셋 기준 모델 결과  
6. segmentation ROI 확장  
7. 부록: 이전 실험 / 아카이브 코드

## 실행 전 확인

이 노트북은 Google Colab + Google Drive 경로를 기준으로 정리되어 있습니다.  
경로 변수는 각 실험 셀 상단에 모아두었고, 학습 결과물은 Drive 내부의 별도 폴더에 저장되도록 정리했습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import io
import math
import time
import shutil
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFilter, ImageEnhance
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid
from torchvision import datasets, transforms
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# A. 데이터 수집 / 데이터셋 구축 / EDA

먼저 현재 데이터셋이 **train / val / test 구조로 잘 정리되어 있는지**,  
그리고 클래스 불균형이 실제로 어느 정도인지부터 확인합니다.

## 데이터셋 구조 및 라벨 확인

이 셀은 `ImageFolder` 기준으로 split별 클래스 구성이 맞는지 확인합니다.  
프로젝트 클래스는 `calculus`, `caries`, `discoloration`, `hypodontia`, `ulcers`의 5개입니다.

In [ ]:
#데이터셋의 라벨링 잘 되어있는지 확인
from torchvision.datasets import ImageFolder

train_ds = ImageFolder("/content/drive/MyDrive/치아질환분류데이터셋/train")
val_ds   = ImageFolder("/content/drive/MyDrive/치아질환분류데이터셋/val")
test_ds  = ImageFolder("/content/drive/MyDrive/치아질환분류데이터셋/test")

print("train class_to_idx:", train_ds.class_to_idx)
print("val   class_to_idx:", val_ds.class_to_idx)
print("test  class_to_idx:", test_ds.class_to_idx)

assert train_ds.class_to_idx == val_ds.class_to_idx == test_ds.class_to_idx, \
    " class_to_idx mismatch! (라벨 매핑이 split마다 다릅니다)"
print(" class_to_idx is consistent.")

## 클래스별 샘플 이미지와 split 분포 확인

이 셀은 각 split / class별 샘플 이미지 일부를 보여주고,  
동시에 클래스 불균형이 실제로 존재하는지 막대그래프로 확인합니다.

In [ ]:
# 이미지를 출력하는 함수
def display_images(image_paths, title, max_images=5):
    plt.figure(figsize=(12, 3))
    for i, image_path in enumerate(image_paths[:max_images]):
        image = Image.open(image_path)
        plt.subplot(1, max_images, i + 1)
        plt.imshow(image)
        plt.axis('off')
        plt.title(title)
    plt.tight_layout()
    plt.show()

gdrive_dataset_path = '/content/drive/MyDrive/치아질환분류데이터셋'

# 이미지와 바 그래프 출력
categories = [
    'Train calculus', 'Train caries', 'Train discoloration', 'Train hypodontia', 'Train ulcers',
    'Val calculus', 'Val caries', 'Val discoloration', 'Val hypodontia', 'Val ulcers',
    'Test calculus', 'Test caries', 'Test discoloration', 'Test hypodontia', 'Test ulcers'
]
paths = [
    f'{gdrive_dataset_path}/train/calculus/*', f'{gdrive_dataset_path}/train/caries/*',
    f'{gdrive_dataset_path}/train/discoloration/*', f'{gdrive_dataset_path}/train/hypodontia/*',
    f'{gdrive_dataset_path}/train/ulcers/*',
    f'{gdrive_dataset_path}/val/calculus/*', f'{gdrive_dataset_path}/val/caries/*',
    f'{gdrive_dataset_path}/val/discoloration/*', f'{gdrive_dataset_path}/val/hypodontia/*',
    f'{gdrive_dataset_path}/val/ulcers/*',
    f'{gdrive_dataset_path}/test/calculus/*', f'{gdrive_dataset_path}/test/caries/*',
    f'{gdrive_dataset_path}/test/discoloration/*', f'{gdrive_dataset_path}/test/hypodontia/*',
    f'{gdrive_dataset_path}/test/ulcers/*'
]

image_counts = []
for category, path in zip(categories, paths):
    image_paths = glob.glob(path)
    image_counts.append(len(image_paths))
    display_images(image_paths, category)

plt.figure(figsize=(15, 5))
plt.bar(categories, image_counts, color=[
    'blue', 'orange', 'green', 'red', 'black',
    'blue', 'orange', 'green', 'red', 'black',
    'blue', 'orange', 'green', 'red', 'black'
])
plt.xlabel('Category')
plt.ylabel('Number of Images')
plt.title('Number of Images per Category')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 평가지표 설정

이 프로젝트는 **불균형 멀티클래스 분류**이므로 단순 accuracy만으로 성능을 판단하면  
소수 클래스 붕괴를 놓치기 쉽습니다. 따라서 아래 기준으로 평가합니다.

- 메인: **Macro-F1**
- 보조: **Balanced Accuracy**
- 필수 보고: **Per-class Recall + Confusion Matrix**

특히 의료·질환 관련 분류에서는 실제 양성을 놓치지 않는지 보기 위해  
클래스별 recall 해석이 중요합니다.

# B. 학습 파이프라인 구축 및 안정화

학습 파이프라인은 **입력 정규화**, **가벼운 증강**, **불균형 완화**,  
그리고 **과적합 감시 기반 early stopping**을 중심으로 구성했습니다.

## 증강 샘플 확인

최종 메인 파이프라인에서는 ImageNet 기반 fine-tuning에 무리가 없는 수준의  
기본 증강만 유지했습니다. 아래 셀은 각 클래스에서 증강 후 이미지가  
질환 단서를 크게 훼손하지 않는지 시각적으로 확인하는 용도입니다.

In [ ]:
# =========================================
# 클래스별(각 5장) "증강 적용된 이미지" 시각화
# - DATA_ROOT: /content/drive/MyDrive/치아질환분류데이터셋
# - split: train (증강은 train에서만 적용하는 것이 일반적)
# =========================================

import os, random
import numpy as np
import torch
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

# (코랩에서 Drive 미마운트면 주석 해제)
# from google.colab import drive
# drive.mount("/content/drive")

# =========================
# 0) 경로/시드
# =========================
DATA_ROOT = "/content/drive/MyDrive/치아질환분류데이터셋"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# =========================
# 1) transforms (질문에 준 그대로)
# =========================
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.0),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

# =========================
# 2) dataset 로드
# =========================
train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
class_names = train_ds.classes
num_classes = len(class_names)

print(" TRAIN_DIR =", TRAIN_DIR)
print(" classes =", class_names)
print(" num_images =", len(train_ds))

# =========================
# 3) denormalize helper (시각화용)
# =========================
mean_t = torch.tensor(mean).view(3, 1, 1)
std_t  = torch.tensor(std).view(3, 1, 1)

def denorm(img_tensor: torch.Tensor) -> torch.Tensor:
    """Normalize된 (C,H,W) 텐서를 [0,1] RGB로 복구 (시각화용)"""
    x = img_tensor.detach().cpu()
    x = x * std_t + mean_t
    x = torch.clamp(x, 0.0, 1.0)
    return x

# =========================
# 4) 클래스별 인덱스 수집
# =========================
targets = [y for _, y in train_ds.samples]  # ImageFolder 내부 레이블
class_to_indices = {c: [] for c in range(num_classes)}
for idx, y in enumerate(targets):
    class_to_indices[y].append(idx)

for ci, cname in enumerate(class_names):
    print(f"- {cname:15s}: {len(class_to_indices[ci])} images")

# =========================
# 5) 클래스별 5장씩 증강 결과 시각화
# =========================
K = 5  # 클래스당 시각화 개수

for ci, cname in enumerate(class_names):
    idxs = class_to_indices[ci]
    if len(idxs) == 0:
        print(f"⚠️ {cname} 클래스에 이미지가 없습니다.")
        continue

    # 가능한 경우 중복 없이 K장, 부족하면 중복 허용
    if len(idxs) >= K:
        picked = random.sample(idxs, K)
    else:
        picked = [random.choice(idxs) for _ in range(K)]

    fig, axes = plt.subplots(1, K, figsize=(3.2 * K, 3.5))
    fig.suptitle(f"[Augmented samples] class = {cname}", fontsize=14)

    for j, ax in enumerate(axes):
        x, y = train_ds[picked[j]]  #  여기서 매번 랜덤 증강이 적용됨
        x_vis = denorm(x)           #  Normalize 되돌림
        img = x_vis.permute(1, 2, 0).numpy()  # (H,W,C)

        ax.imshow(img)
        ax.axis("off")
        ax.set_title(f"idx={picked[j]}", fontsize=10)

    plt.tight_layout()
    plt.show()

## 시드 고정

실험 간 비교를 가능하게 하려면 초기화·셔플·증강에 들어가는 난수를 통제할 필요가 있습니다.  
이 셀은 재현성을 위한 기본 시드 설정 함수입니다.

In [ ]:
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)
print(f"Random seed set to {SEED}")

## 학습 / 평가 유틸리티

이 블록에는 다음이 포함됩니다.

- confusion matrix 기반 metric 계산
- watch-on 기반 2단계 early stopping
- VGG / ResNet / DenseNet 공통 학습 루프
- Macro-F1 / Balanced Accuracy / Per-class Recall 기록
- best checkpoint 저장 / 복원

In [ ]:
#수정된 함수정의 코드(각 모델별 효과좋은  fine-tuning)
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import numpy as np

# =========================
# confusion matrix 기반 지표 계산 함수 (기존 유지)
# =========================
def metrics_from_confusion_matrix(cm: torch.Tensor):
    cm_f = cm.to(dtype=torch.float32)

    tp = torch.diag(cm_f)
    fn = cm_f.sum(dim=1) - tp
    fp = cm_f.sum(dim=0) - tp

    eps = 1e-12
    recall = tp / (tp + fn + eps)
    precision = tp / (tp + fp + eps)

    balanced_acc = recall.mean().item()

    f1 = 2 * precision * recall / (precision + recall + eps)
    macro_f1 = f1.mean().item()

    per_class_recall = recall.detach().cpu().tolist()
    return macro_f1, balanced_acc, per_class_recall


class BestTracker:
    def __init__(self, mode: str, min_delta: float, name: str):
        assert mode in ["min", "max"]
        self.mode = mode
        self.min_delta = float(min_delta)
        self.name = name
        self.best = None
        self.best_epoch = None

    def improved(self, x: float) -> bool:
        if self.best is None:
            return True
        if self.mode == "min":
            return x < (self.best - self.min_delta)
        else:
            return x > (self.best + self.min_delta)

    def update(self, x: float, epoch: int):
        x = float(x)
        prev = self.best
        if self.improved(x):
            self.best = x
            self.best_epoch = epoch
            return True, prev
        return False, prev


#  ckpt 저장 키 호환성 강화: model / model_state_dict 둘 다 저장
def save_ckpt(path, epoch, model, optimizer, scheduler, best_val_loss, best_macro_f1,
              best_bal_acc, best_per_class_recall, class_names):
    sd = model.state_dict()
    torch.save({
        "epoch": epoch,
        "model": sd,  # (기존 키)
        "model_state_dict": sd,  #  호환 키
        "optimizer": optimizer.state_dict() if optimizer else None,
        "scheduler": scheduler.state_dict() if scheduler else None,
        "best_val_loss": best_val_loss,
        "best_macro_f1": best_macro_f1,
        "best_balanced_acc": best_bal_acc,
        "best_per_class_recall": best_per_class_recall,
        "class_names": class_names,
    }, path)


def load_ckpt(path, model, optimizer=None, scheduler=None, device="cpu"):
    ckpt = torch.load(path, map_location=device)
    #  model_state_dict 우선, 없으면 model 키 사용
    if "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"])
    else:
        model.load_state_dict(ckpt["model"])
    if optimizer is not None and ckpt.get("optimizer") is not None:
        optimizer.load_state_dict(ckpt["optimizer"])
    if scheduler is not None and ckpt.get("scheduler") is not None:
        scheduler.load_state_dict(ckpt["scheduler"])
    return ckpt


# =========================================================
# CutMix/Mixup 헬퍼 (Train에서만 사용)
# =========================================================
def rand_bbox(W, H, lam):
    cut_rat = np.sqrt(1.0 - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    return x1, y1, x2, y2


def cutmix_data(x, y, alpha=1.0):
    if alpha <= 0:
        return x, y, y, 1.0

    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0), device=x.device)

    W = x.size(3)
    H = x.size(2)
    x1, y1, x2, y2 = rand_bbox(W, H, lam)

    x_cut = x.clone()
    x_cut[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]

    patch_area = (x2 - x1) * (y2 - y1)
    lam = 1.0 - patch_area / float(W * H)

    y_a, y_b = y, y[index]
    return x_cut, y_a, y_b, lam


def mixup_data(x, y, alpha=0.2):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0), device=x.device)
    x_mix = lam * x + (1.0 - lam) * x[index]
    y_a, y_b = y, y[index]
    return x_mix, y_a, y_b, lam


def mixed_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1.0 - lam) * criterion(pred, y_b)


# =========================
#  train_model_v2: optimizer 외부 주입 가능
# =========================
def train_model_v2(
    optimizer_name,
    net,
    train_loader,
    val_loader,
    criterion,
    num_epochs,
    scheduler=None,
    ckpt_path="best_v3.pt",
    start_epoch=0,
    device=None,
    print_per_class_recall=True,

    warmup_epochs=50,
    watch_patience=6,
    overfit_patience=6,

    loss_min_delta=0.002,
    f1_min_delta=0.002,

    f1_worsen_delta=0.005,
    f1_worsen_patience=2,

    stop_on_overfit=True,

    #  CutMix/Mixup 옵션
    use_cutmix=False,
    cutmix_prob=0.5,
    cutmix_alpha=1.0,
    use_mixup=False,
    mixup_alpha=0.2,

    # optimizer 외부 주입(모델별 최적 튜닝을 위해)
    optimizer=None,
    default_lr=3e-5,
    default_weight_decay=5e-4,
    default_momentum=0.9,
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net.to(device)

    # optimizer가 들어오면 그대로 사용, 없으면 기존 방식대로 생성
    if optimizer is None:
        if optimizer_name == "SGD":
            optimizer = optim.SGD(net.parameters(), lr=default_lr, momentum=default_momentum, weight_decay=default_weight_decay)
        elif optimizer_name == "Adam":
            optimizer = optim.Adam(net.parameters(), lr=default_lr, betas=(0.9, 0.999))
        elif optimizer_name == "RAdam":
            optimizer = optim.RAdam(net.parameters(), lr=default_lr, betas=(0.9, 0.999))
        else:
            raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    class_names = train_loader.dataset.classes
    num_classes = len(class_names)

    train_losses, val_losses = [], []
    val_accuracies, val_macro_f1s, val_balanced_accuracies = [], [], []
    val_per_class_recalls = []
    gap_history = []

    val_loss_tracker = BestTracker(mode="min", min_delta=loss_min_delta, name="val_loss")
    f1_tracker = BestTracker(mode="max", min_delta=f1_min_delta, name="macro_f1")

    watch_on = False
    no_improve_val_loss = 0
    watch_no_improve_f1 = 0
    watch_worse_f1_streak = 0

    best_val_loss_for_ckpt = None
    best_f1_for_ckpt = None
    best_bal_acc_for_ckpt = None
    best_recall_for_ckpt = None

    for epoch in range(start_epoch, num_epochs):
        # -------- Train --------
        net.train()
        running_loss = 0.0
        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [train]"):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            if use_cutmix and (np.random.rand() < cutmix_prob):
                inputs_m, y_a, y_b, lam = cutmix_data(inputs, labels, alpha=cutmix_alpha)
                outputs = net(inputs_m)
                loss = mixed_criterion(criterion, outputs, y_a, y_b, lam)
            elif use_mixup:
                inputs_m, y_a, y_b, lam = mixup_data(inputs, labels, alpha=mixup_alpha)
                outputs = net(inputs_m)
                loss = mixed_criterion(criterion, outputs, y_a, y_b, lam)
            else:
                outputs = net(inputs)
                loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        train_loss = running_loss / max(1, len(train_loader))
        train_losses.append(train_loss)

        # -------- Val --------
        net.eval()
        val_loss_sum = 0.0
        correct, total = 0, 0
        cm = torch.zeros((num_classes, num_classes), device=device, dtype=torch.int64)

        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [val]"):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = net(inputs)
                loss = criterion(outputs, labels)
                val_loss_sum += loss.item()

                predicted = outputs.argmax(dim=1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

                idx = labels * num_classes + predicted
                cm += torch.bincount(idx, minlength=num_classes * num_classes).reshape(num_classes, num_classes)

        val_loss = val_loss_sum / max(1, len(val_loader))
        val_losses.append(val_loss)

        val_acc = 100.0 * correct / max(1, total)
        val_accuracies.append(val_acc)

        val_macro_f1, val_bal_acc, per_class_recall = metrics_from_confusion_matrix(cm)
        val_macro_f1s.append(val_macro_f1)
        val_balanced_accuracies.append(val_bal_acc)
        val_per_class_recalls.append(per_class_recall)

        gap_history.append(max(0.0, float(val_loss - train_loss)))

        print(
            f'[{optimizer_name}] Epoch {epoch+1}, '
            f'Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}, '
            f'Val Acc: {val_acc:.2f}%, Val Macro-F1: {val_macro_f1:.4f}, '
            f'Val Balanced Acc: {val_bal_acc:.4f}'
        )

        if print_per_class_recall:
            recall_str = " | ".join([f"{n}:{r:.3f}" for n, r in zip(class_names, per_class_recall)])
            print(f"Per-class Recall: {recall_str}")

        # best 갱신 로그
        val_loss_improved, _ = val_loss_tracker.update(val_loss, epoch)
        if val_loss_improved:
            print(f" BEST val_loss 업데이트: {val_loss_tracker.best:.6f} (epoch {epoch+1})")
            no_improve_val_loss = 0
        else:
            if (epoch + 1) > warmup_epochs:
                no_improve_val_loss += 1

        f1_improved, _ = f1_tracker.update(val_macro_f1, epoch)
        if f1_improved:
            print(f" BEST macro-F1 업데이트: {f1_tracker.best:.4f} (epoch {epoch+1})")

            best_val_loss_for_ckpt = val_loss_tracker.best
            best_f1_for_ckpt = f1_tracker.best
            best_bal_acc_for_ckpt = val_bal_acc
            best_recall_for_ckpt = per_class_recall

            save_ckpt(
                ckpt_path, epoch, net, optimizer, scheduler,
                best_val_loss_for_ckpt, best_f1_for_ckpt,
                best_bal_acc_for_ckpt, best_recall_for_ckpt, class_names
            )

        if (epoch + 1) <= warmup_epochs:
            if scheduler is not None:
                try:
                    scheduler.step(val_loss)
                except TypeError:
                    scheduler.step()
            continue

        if (not watch_on) and (no_improve_val_loss >= watch_patience):
            watch_on = True
            watch_no_improve_f1 = 0
            watch_worse_f1_streak = 0
            print(f"🟡 WATCH ON: val_loss best 갱신 없음 {watch_patience} epochs (warmup={warmup_epochs} 이후)")

        if watch_on and val_loss_improved:
            watch_on = False
            watch_no_improve_f1 = 0
            watch_worse_f1_streak = 0
            print("🟢 WATCH OFF: val_loss가 다시 best 갱신되어 경보 해제")

        overfit_trigger = False
        if watch_on:
            if not f1_improved:
                watch_no_improve_f1 += 1
            else:
                watch_no_improve_f1 = 0

            if (f1_tracker.best is not None) and (val_macro_f1 < (f1_tracker.best - f1_worsen_delta)):
                watch_worse_f1_streak += 1
            else:
                watch_worse_f1_streak = 0

            if (watch_no_improve_f1 >= overfit_patience) or (watch_worse_f1_streak >= f1_worsen_patience):
                overfit_trigger = True

        if overfit_trigger:
            print(
                f"🔴 OVERFIT ON: WATCH 중 macro-F1 정체/악화 → 종료 (epoch {epoch+1}) "
                f"(no_improve_f1_in_watch={watch_no_improve_f1}/{overfit_patience}, "
                f"worse_streak={watch_worse_f1_streak}/{f1_worsen_patience})"
            )
            if stop_on_overfit:
                break

        if scheduler is not None:
            try:
                scheduler.step(val_loss)
            except TypeError:
                scheduler.step()

    ckpt = load_ckpt(ckpt_path, net, optimizer=optimizer, scheduler=scheduler, device=device)
    best_epoch = ckpt["epoch"] + 1

    return (
        train_losses, val_losses, val_accuracies,
        val_macro_f1s, val_balanced_accuracies, val_per_class_recalls,
        gap_history,
        best_epoch,
        ckpt["best_macro_f1"],
        ckpt["best_balanced_acc"],
        ckpt["class_names"],
        ckpt["best_per_class_recall"],
        ckpt["best_val_loss"],
    )

In [ ]:
# (선택) 그래프에서 한글이 깨질 때만 사용
from matplotlib import font_manager

candidate_fonts = [
    "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
    "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
]

for font_path in candidate_fonts:
    if os.path.exists(font_path):
        try:
            font_manager.fontManager.addfont(font_path)
            plt.rcParams["font.family"] = font_manager.FontProperties(fname=font_path).get_name()
            plt.rcParams["axes.unicode_minus"] = False
            print("using font:", font_path)
            break
        except Exception as e:
            print("font setup skipped:", repr(e))

# C. 원본 데이터셋 기준 모델 학습 결과

이 섹션은 **원본 train / val / test split**을 그대로 사용했을 때의 성능을 확인하는 구간입니다.  
VGG19_BN, ResNet50, DenseNet121을 동일한 평가 기준으로 비교하고,  
이후 `discoloration` 클래스 중심 Grad-CAM을 확인합니다.

## 원본 데이터셋 학습 / 평가 실행

In [ ]:
# ============================================================
#  결과/체크포인트를 무조건 Google Drive에 저장하도록 수정
# - best_*.pt + report_onepage.pdf/png -> Drive 경로에 바로 저장
# - 런타임 끊겨도 파일 보존
# ============================================================

import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision.models import VGG19_BN_Weights, ResNet50_Weights, DenseNet121_Weights
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler

import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# 데이터 루트 스위치
# ============================================================
DATA_ROOT = "/content/drive/MyDrive/치아질환분류데이터셋"  #  원본
# DATA_ROOT = "/content/drive/MyDrive/치아질환분류데이터셋_resplit_classwise_v1"  # 재분할

# ============================================================
#  결과/ckpt/리포트는 Drive에 바로 저장
# ============================================================
OUT_DIR = "/content/drive/MyDrive/model_compare_bestft_fixedclass/orig"
os.makedirs(OUT_DIR, exist_ok=True)
print(" ORIG OUT_DIR =", OUT_DIR)

# ============================================================
#  클래스 순서 고정
# ============================================================
FIXED_CLASSES = ["calculus", "caries", "discoloration", "hypodontia", "ulcers"]

class RemapTargetsDataset(torch.utils.data.Dataset):
    def __init__(self, base_ds, fixed_classes):
        self.base = base_ds
        self.fixed_classes = list(fixed_classes)
        self.fixed_class_to_idx = {c:i for i,c in enumerate(self.fixed_classes)}

        ds_set = set(base_ds.classes)
        fixed_set = set(self.fixed_classes)
        if ds_set != fixed_set:
            missing = sorted(list(fixed_set - ds_set))
            extra = sorted(list(ds_set - fixed_set))
            raise ValueError(f"[class mismatch] missing={missing}, extra={extra}. base_ds.classes={base_ds.classes}")

        inv_old = {old_idx: cls for cls, old_idx in base_ds.class_to_idx.items()}
        self.old_to_fixed = {old_idx: self.fixed_class_to_idx[inv_old[old_idx]] for old_idx in inv_old}

        self.classes = self.fixed_classes
        self.class_to_idx = self.fixed_class_to_idx
        self.targets = [self.old_to_fixed[int(t)] for t in base_ds.targets]

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        x, y_old = self.base[idx]
        y = self.old_to_fixed[int(y_old)]
        return x, y

# ============================================================
# transforms
# ============================================================
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.0),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

eval_transforms = transforms.Compose([
    transforms.Resize(size=(256)),
    transforms.CenterCrop(size=(224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# ============================================================
# dataset 로드 (raw -> fixed mapping)
# ============================================================
train_ds_raw = datasets.ImageFolder(f"{DATA_ROOT}/train", transform=train_transforms)
val_ds_raw   = datasets.ImageFolder(f"{DATA_ROOT}/val",   transform=eval_transforms)
test_ds_raw  = datasets.ImageFolder(f"{DATA_ROOT}/test",  transform=eval_transforms)

train_ds = RemapTargetsDataset(train_ds_raw, FIXED_CLASSES)
val_ds   = RemapTargetsDataset(val_ds_raw, FIXED_CLASSES)
test_ds  = RemapTargetsDataset(test_ds_raw, FIXED_CLASSES)

class_names = train_ds.classes
num_classes = len(class_names)

print("DATA_ROOT:", DATA_ROOT)
print("FIXED class order:", class_names)
print("fixed class_to_idx:", train_ds.class_to_idx)

# ============================================================
# sampler 유지
# ============================================================
labels = torch.tensor(train_ds.targets, dtype=torch.long)
class_count = torch.bincount(labels, minlength=num_classes).float().clamp_min(1.0)
alpha = 1.0
class_weight = (1.0 / class_count) ** alpha
sample_weight = class_weight[labels]

SEED = 42
g = torch.Generator(); g.manual_seed(SEED)

train_sampler = WeightedRandomSampler(
    weights=sample_weight,
    num_samples=len(sample_weight),
    replacement=True,
    generator=g
)

train_loader = DataLoader(train_ds, batch_size=32, sampler=train_sampler, shuffle=False, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

# ============================================================
# 모델별 fine-tuning 정책
# ============================================================
def freeze_all(net):
    for p in net.parameters():
        p.requires_grad = False

def enable_bn_affine(net):
    for m in net.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            if m.weight is not None: m.weight.requires_grad = True
            if m.bias is not None:   m.bias.requires_grad = True

def build_vgg19_bn(num_classes):
    net = models.vgg19_bn(weights=VGG19_BN_Weights.DEFAULT)
    freeze_all(net)
    enable_bn_affine(net)
    net.classifier[6] = nn.Linear(4096, num_classes)
    for p in net.classifier[6].parameters():
        p.requires_grad = True
    return net

def build_resnet50_bestft(num_classes):
    net = models.resnet50(weights=ResNet50_Weights.DEFAULT)
    freeze_all(net)
    for p in net.layer4.parameters():
        p.requires_grad = True
    enable_bn_affine(net)
    net.fc = nn.Linear(net.fc.in_features, num_classes)
    for p in net.fc.parameters():
        p.requires_grad = True
    return net

def build_densenet121_bestft(num_classes):
    net = models.densenet121(weights=DenseNet121_Weights.DEFAULT)
    freeze_all(net)
    for p in net.features.denseblock4.parameters():
        p.requires_grad = True
    if hasattr(net.features, "norm5"):
        for p in net.features.norm5.parameters():
            p.requires_grad = True
    enable_bn_affine(net)
    net.classifier = nn.Linear(net.classifier.in_features, num_classes)
    for p in net.classifier.parameters():
        p.requires_grad = True
    return net

def make_sgd_param_groups(net, head_keys, backbone_keys,
                          lr_head=1e-3, lr_backbone=1e-4, lr_bn=3e-4,
                          weight_decay=5e-4, momentum=0.9):
    head_params, backbone_params, bn_params = [], [], []
    for name, p in net.named_parameters():
        if not p.requires_grad:
            continue
        if "bn" in name.lower() or "batchnorm" in name.lower():
            bn_params.append(p)
            continue
        if any(k in name for k in head_keys):
            head_params.append(p)
        elif any(k in name for k in backbone_keys):
            backbone_params.append(p)
        else:
            backbone_params.append(p)

    param_groups = []
    if head_params:
        param_groups.append({"params": head_params, "lr": lr_head, "weight_decay": weight_decay, "momentum": momentum})
    if backbone_params:
        param_groups.append({"params": backbone_params, "lr": lr_backbone, "weight_decay": weight_decay, "momentum": momentum})
    if bn_params:
        param_groups.append({"params": bn_params, "lr": lr_bn, "weight_decay": 0.0, "momentum": momentum})

    return optim.SGD(param_groups, lr=lr_head, momentum=momentum, weight_decay=weight_decay)

# ============================================================
# TEST 평가
# ============================================================
@torch.no_grad()
def evaluate_on_loader(net, loader, criterion, num_classes, device):
    net.eval()
    cm = torch.zeros((num_classes, num_classes), dtype=torch.int64)
    total_loss, total_n = 0.0, 0

    for x, y in loader:
        x = x.to(device); y = y.to(device)
        logits = net(x)
        loss = criterion(logits, y)
        bs = y.size(0)
        total_loss += float(loss.item()) * bs
        total_n += bs

        pred = logits.argmax(dim=1)
        for t, p in zip(y.view(-1), pred.view(-1)):
            cm[t.long(), p.long()] += 1

    cm_f = cm.float()
    tp = torch.diag(cm_f)
    fn = cm_f.sum(dim=1) - tp
    fp = cm_f.sum(dim=0) - tp

    eps = 1e-12
    recall = tp / (tp + fn + eps)
    precision = tp / (tp + fp + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)

    macro_f1 = float(f1.mean().item())
    balanced_acc = float(recall.mean().item())
    acc = float(tp.sum().item() / max(cm_f.sum().item(), 1.0))
    avg_loss = total_loss / max(total_n, 1)

    return cm.cpu().numpy(), recall.cpu().numpy(), acc, macro_f1, balanced_acc, avg_loss

def normalize_rows(cm):
    row_sum = cm.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1
    return cm / row_sum

criterion = nn.CrossEntropyLoss()

# ============================================================
# 학습 + 리포트 생성
# ============================================================
MODELS = [
    ("vgg19_bn", build_vgg19_bn),
    ("resnet50", build_resnet50_bestft),
    ("densenet121", build_densenet121_bestft),
]

results = []

for name, builder in MODELS:
    print("\n" + "="*80)
    print("Training:", name)
    print("="*80)

    net = builder(num_classes).to(device)

    if name == "vgg19_bn":
        optimizer = None
    elif name == "resnet50":
        optimizer = make_sgd_param_groups(net, head_keys=["fc."], backbone_keys=["layer4."],
                                          lr_head=1e-3, lr_backbone=1e-4, lr_bn=3e-4)
    else:
        optimizer = make_sgd_param_groups(net, head_keys=["classifier."],
                                          backbone_keys=["features.denseblock4.", "features.norm5."],
                                          lr_head=1e-3, lr_backbone=1e-4, lr_bn=3e-4)

    ckpt_path = os.path.join(OUT_DIR, f"best_{name}.pt")

    train_model_v2(
        optimizer_name="SGD",
        net=net,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        num_epochs=100,
        device=device,
        ckpt_path=ckpt_path,
        use_cutmix=False,
        use_mixup=False,
        optimizer=optimizer,
        default_lr=3e-5
    )

    ckpt = torch.load(ckpt_path, map_location=device)
    if "model_state_dict" in ckpt:
        net.load_state_dict(ckpt["model_state_dict"], strict=True)
    else:
        net.load_state_dict(ckpt["model"], strict=True)

    cm, recall, acc, macro_f1, bal_acc, loss = evaluate_on_loader(net, test_loader, criterion, num_classes, device)
    cm_norm = normalize_rows(cm)

    #  Test per-class recall 콘솔 출력
    recall_str = " | ".join([f"{cls}:{recall[i]:.3f}" for i, cls in enumerate(class_names)])

    results.append({
        "model": name,
        "cm_norm": cm_norm,
        "recall": recall,
        "acc": acc,
        "macro_f1": macro_f1,
        "bal_acc": bal_acc,
        "loss": loss
    })

    print(f"[{name}] TEST loss={loss:.4f} | acc={acc*100:.2f}% | macroF1={macro_f1:.4f} | bal_acc={bal_acc*100:.2f}%")
    print(f"[{name}] TEST Per-class Recall: {recall_str}")

# 1장 리포트 (Drive 저장)
table_cols = ["Class"] + [r["model"] for r in results]
table_data = []
for i, cls in enumerate(class_names):
    table_data.append([cls] + [f"{r['recall'][i]:.3f}" for r in results])

overall_cols = ["Metric"] + [r["model"] for r in results]
overall_data = [
    ["Test Loss"] + [f"{r['loss']:.4f}" for r in results],
    ["Test Acc"] + [f"{r['acc']*100:.2f}%" for r in results],
    ["Test Macro-F1"] + [f"{r['macro_f1']:.4f}" for r in results],
    ["Test Balanced Acc"] + [f"{r['bal_acc']*100:.2f}%" for r in results],
]

fig = plt.figure(figsize=(18, 11))
for idx, r in enumerate(results):
    ax = fig.add_subplot(2, 3, idx+1)
    im = ax.imshow(r["cm_norm"], aspect="auto")
    ax.set_title(f"{r['model']} (normalized CM)")
    ax.set_xlabel("Pred"); ax.set_ylabel("True")
    ax.set_xticks(range(num_classes)); ax.set_yticks(range(num_classes))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)

    for i in range(num_classes):
        for j in range(num_classes):
            ax.text(j, i, f"{r['cm_norm'][i,j]:.2f}", ha="center", va="center", fontsize=7)

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax_tbl = fig.add_subplot(2, 1, 2)
ax_tbl.axis("off")

tbl1 = ax_tbl.table(cellText=table_data, colLabels=table_cols, loc="upper center")
tbl1.auto_set_font_size(False); tbl1.set_fontsize(10); tbl1.scale(1.0, 1.25)

tbl2 = ax_tbl.table(cellText=overall_data, colLabels=overall_cols, loc="lower center")
tbl2.auto_set_font_size(False); tbl2.set_fontsize(11); tbl2.scale(1.0, 1.35)

ax_tbl.set_title("TEST Report: Per-class Recall + Overall Metrics (fixed class order)", pad=10)

plt.tight_layout()
pdf_path = os.path.join(OUT_DIR, "report_onepage.pdf")
png_path = os.path.join(OUT_DIR, "report_onepage.png")
plt.savefig(pdf_path)
plt.savefig(png_path, dpi=220)
plt.close()

print("\n Saved report to Drive:")
print(" -", pdf_path)
print(" -", png_path)
print(" Saved ckpts to Drive (best_*.pt) inside:", OUT_DIR)

## 원본 데이터셋 Grad-CAM 비교

원본 split에서는 특히 `discoloration` 클래스가 다른 클래스 대비 불안정하게 분류되는지  
Grad-CAM으로 모델이 보는 부분을 함께 확인합니다.

In [ ]:
# =========================
# Grad-CAM 시각화: 원본 + 모델별 Grad-CAM 비교 그리드
# =========================
from pathlib import Path
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image
import matplotlib.pyplot as plt

from torchvision import transforms
from torchvision.datasets import ImageFolder
import torchvision.models as models

# =========================================
# 0) Config
# =========================================
DATA_ROOT = "/content/drive/MyDrive/치아질환분류데이터셋"
TEST_DIR = f"{DATA_ROOT}/test"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)

MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

IMG_SIZE = 224
SAVE_DIR = "/content/drive/MyDrive/model_compare_bestft_fixedclass/orig/gradcam_compare"
os.makedirs(SAVE_DIR, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================
# 1) Dataset
# =========================================
test_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

test_ds = ImageFolder(TEST_DIR, transform=test_tf)
class_to_idx = test_ds.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}

def find_discoloration_idx(class_to_idx: dict):
    for k, v in class_to_idx.items():
        if k.lower() == "discoloration":
            return v, k
    for k, v in class_to_idx.items():
        if "discolor" in k.lower():
            return v, k
    raise ValueError(f"Cannot find discoloration class in: {list(class_to_idx.keys())}")

DIS_IDX, DIS_NAME = find_discoloration_idx(class_to_idx)
print(" Discoloration class:", DIS_NAME, "->", DIS_IDX)

dis_samples = [(i, p, y) for i, (p, y) in enumerate(test_ds.samples) if y == DIS_IDX]
print(" Discoloration test samples:", len(dis_samples))
assert len(dis_samples) > 0, "test에 discoloration 샘플이 0개입니다."

# =========================================
# 2) Grad-CAM
# =========================================
def find_last_conv_layer(model: nn.Module):
    last_name, last_module = None, None
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            last_name, last_module = name, module
    if last_module is None:
        raise ValueError("No Conv2d layer found.")
    return last_name, last_module

class GradCAM:
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None

        def fwd_hook(module, inp, out):
            self.activations = out

        def bwd_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0]

        self.h1 = self.target_layer.register_forward_hook(fwd_hook)
        self.h2 = self.target_layer.register_full_backward_hook(bwd_hook)

    def remove_hooks(self):
        self.h1.remove()
        self.h2.remove()

    @torch.enable_grad()
    def generate(self, x: torch.Tensor, target_class: int):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        score = logits[:, target_class].sum()
        score.backward(retain_graph=True)

        A = self.activations
        dYdA = self.gradients

        weights = dYdA.mean(dim=(2,3), keepdim=True)
        cam = (weights * A).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(x.shape[2], x.shape[3]), mode="bilinear", align_corners=False)

        cam = cam.squeeze().detach()
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-12)
        return cam, logits.detach()

def unnormalize(img_t: torch.Tensor, mean=MEAN, std=STD):
    mean = torch.tensor(mean, device=img_t.device).view(3,1,1)
    std = torch.tensor(std, device=img_t.device).view(3,1,1)
    x = img_t * std + mean
    x = x.clamp(0,1)
    return x.permute(1,2,0).cpu().numpy()

def overlay_cam(rgb01: np.ndarray, cam01: np.ndarray, alpha=0.45):
    cmap = plt.get_cmap("jet")
    heat = cmap(cam01)[...,:3]
    out = (1 - alpha) * rgb01 + alpha * heat
    return np.clip(out, 0, 1)

# =========================================
# 3) Model build/load
# =========================================
def build_model(arch: str, num_classes: int):
    arch = arch.lower()
    if arch == "vgg19_bn":
        m = models.vgg19_bn(weights=None)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
        return m
    if arch == "resnet50":
        m = models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m
    if arch == "densenet121":
        m = models.densenet121(weights=None)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
        return m
    raise ValueError(f"Unknown arch: {arch}")

def load_checkpoint(model: nn.Module, ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        sd = ckpt["model_state_dict"]
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        sd = ckpt["state_dict"]
    elif isinstance(ckpt, dict):
        sd = ckpt
    else:
        raise ValueError("Unsupported checkpoint format.")

    new_sd = {k.replace("module.",""): v for k, v in sd.items()}
    model.load_state_dict(new_sd, strict=True)
    return model

MODEL_SPECS = [
    {"name": "VGG19_BN", "arch": "vgg19_bn", "ckpt": "/content/drive/MyDrive/model_compare_bestft_fixedclass/orig/best_vgg19_bn.pt"},
    {"name": "ResNet50", "arch": "resnet50", "ckpt": "/content/drive/MyDrive/model_compare_bestft_fixedclass/orig/best_resnet50.pt"},
    {"name": "DenseNet121", "arch": "densenet121", "ckpt": "/content/drive/MyDrive/model_compare_bestft_fixedclass/orig/best_densenet121.pt"},
]

NUM_CLASSES = len(class_to_idx)

#  모델들 미리 로드 + 각 모델별 Grad-CAM 준비
models_dict = {}
for spec in MODEL_SPECS:
    m = build_model(spec["arch"], NUM_CLASSES)
    m = load_checkpoint(m, spec["ckpt"])
    m = m.to(device).eval()

    layer_name, layer_module = find_last_conv_layer(m)
    print(f" {spec['name']} target layer:", layer_name)
    cam_engine = GradCAM(m, layer_module)

    models_dict[spec["name"]] = {"model": m, "cam": cam_engine}

# =========================================
# 4) 한 화면 비교 시각화 (rows=images, cols=원본+모델들)
# =========================================
alpha = 0.45
target_class = DIS_IDX

n_imgs = len(dis_samples)          # 18 예상
n_models = len(MODEL_SPECS)
n_cols = n_models + 1              #  원본 컬럼(1개) 추가

#  figure 가로폭도 컬럼 수에 맞춰 확장
fig = plt.figure(figsize=(n_cols * 4.6, n_imgs * 2.6))

for r, (idx, img_path, y) in enumerate(dis_samples):
    x, _ = test_ds[idx]
    x1 = x.unsqueeze(0).to(device)
    rgb01 = unnormalize(x.to(device))

    base = os.path.splitext(os.path.basename(img_path))[0]

    # -------------------------
    # (A) 원본 이미지 컬럼
    # -------------------------
    ax0 = plt.subplot(n_imgs, n_cols, r * n_cols + 1)   # col=0
    ax0.imshow(rgb01)
    ax0.axis("off")
    if r == 0:
        ax0.set_title("Original", fontsize=10)          #  첫 줄 헤더

    #  이미지 id 표시는 원본 칸에 표시(가장 직관적)
    ax0.text(3, 12, base, color="white", fontsize=8,
             bbox=dict(facecolor="black", alpha=0.55, pad=2))

    # -------------------------
    # (B) 모델별 Grad-CAM 컬럼들
    # -------------------------
    for m_i, spec in enumerate(MODEL_SPECS):
        name = spec["name"]
        m = models_dict[name]["model"]
        cam_engine = models_dict[name]["cam"]

        with torch.no_grad():
            logits0 = m(x1)
            prob = F.softmax(logits0, dim=1)[0]
            pred = int(torch.argmax(prob).item())
            pred_name = idx_to_class[pred]
            pred_p = float(prob[pred].item())
            dis_p = float(prob[DIS_IDX].item())
            correct = (pred == DIS_IDX)

        cam01, _ = cam_engine.generate(x1, target_class=target_class)
        over = overlay_cam(rgb01, cam01.cpu().numpy(), alpha=alpha)

        #  col index: 원본(0) 다음이 첫 모델(1)
        col = 1 + m_i
        ax = plt.subplot(n_imgs, n_cols, r * n_cols + col + 1)
        ax.imshow(over)
        ax.axis("off")

        # 첫 줄(row=0)에만 모델 이름 헤더처럼
        if r == 0:
            ax.set_title(f"{name}\nPred={pred_name}({pred_p:.2f}) P(dis)={dis_p:.2f}", fontsize=9)
        else:
            ax.set_title(f"Pred={pred_name}({pred_p:.2f}) P(dis)={dis_p:.2f} | ok={correct}", fontsize=8)

grid_path = os.path.join(SAVE_DIR, "GRID__ORIG_plus_compare_VGG_ResNet_DenseNet__target_discoloration.png")  #  파일명
plt.tight_layout()
plt.savefig(grid_path, dpi=200)
plt.show()

# hook 해제
for name in models_dict:
    models_dict[name]["cam"].remove_hooks()

print("saved:", grid_path)

In [ ]:
from IPython.display import Image as IPyImage, display, IFrame
from pathlib import Path

ORIG_OUT_DIR = Path("/content/drive/MyDrive/model_compare_bestft_fixedclass/orig")
png_path = ORIG_OUT_DIR / "report_onepage.png"
pdf_path = ORIG_OUT_DIR / "report_onepage.pdf"

print("PNG exists?", png_path.exists(), png_path)
print("PDF exists?", pdf_path.exists(), pdf_path)

if png_path.exists():
    display(IPyImage(filename=str(png_path)))
if pdf_path.exists():
    display(IFrame(src=str(pdf_path), width=1100, height=750))

# D. 데이터 재분할 원인 분석 및 sharpness 기반 재분할

원본 split은 웹 크롤링 후 순서 기반으로 나뉜 데이터라서,  
train / val / test 간 입력 분포 차이가 실제 일반화 성능을 왜곡할 가능성이 큽니다.

이 섹션에서는 다음 순서로 원인을 추적합니다.

1. no-reference 품질 proxy 추출  
2. sharpness 기반 class-wise resplit 생성  
3. 원본 vs 재분할의 입력 분포 차이 정량 비교

## 원본 split 품질 proxy 추출

blur / brightness / contrast / saturation / noise / high-frequency energy / file size를 기준으로  
현재 원본 split의 품질 분포를 먼저 확인합니다.

In [ ]:
# ============================================
#  Colab 1-cell: train/val/test 품질(blur/brightness/contrast/noise 등) 정량 비교
# - no-reference 품질 proxy 지표 추출
# - split별 요약 통계 + CSV 저장
# - "test가 더 구리다"를 수치로 증명하는 용도
# ============================================



from pathlib import Path
import os, random, math
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import cv2

# =========================
# 설정
# =========================
DATA_ROOT = Path("/content/drive/MyDrive/치아질환분류데이터셋")  #  경로
SPLITS = ["train", "val", "test"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# 속도/대표성 trade-off: split당 샘플 수 (0이면 전부)
MAX_PER_SPLIT = 2410   # 추천: 300~1200 (너 데이터 크기 고려)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OUT_DIR = Path("/content/drive/MyDrive/quality_shift_check_orig")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# 유틸
# =========================
def iter_images(root: Path):
    paths = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            paths.append(p)
    return paths

def robust_mad(x: np.ndarray, eps=1e-12):
    # Median Absolute Deviation -> robust std proxy
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    return 1.4826 * mad + eps

def compute_metrics(img_rgb: np.ndarray):
    """
    no-reference 품질 proxy metrics
    - sharpness_lapvar: Laplacian variance (클수록 선명)
    - brightness_mean: 밝기 평균(0~255)
    - contrast_std: 밝기 표준편차(대비 proxy)
    - saturation_mean: 채도 평균(0~255)
    - noise_sigma: 고주파 성분의 robust std (노이즈/질감 proxy)
    - hf_energy_ratio: 고주파 에너지 비율(압축/블러에 민감)
    """
    # RGB -> Gray
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    # 1) 선명도(blur) proxy: Laplacian variance
    lap = cv2.Laplacian(gray, cv2.CV_32F, ksize=3)
    sharpness_lapvar = float(lap.var())

    # 2) 밝기/대비
    brightness_mean = float(gray.mean())
    contrast_std = float(gray.std())

    # 3) 채도
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    saturation_mean = float(hsv[..., 1].mean())

    # 4) 노이즈 proxy: (gray - gaussian_blur)의 robust std
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    highpass = (gray.astype(np.float32) - blur.astype(np.float32)).flatten()
    noise_sigma = float(robust_mad(highpass))

    # 5) 고주파 에너지 비율: Sobel magnitude 기반
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    mag = cv2.magnitude(gx, gy)
    hf_energy = float(np.mean(mag))
    total_energy = float(np.mean(np.abs(gray.astype(np.float32))) + 1e-6)
    hf_energy_ratio = hf_energy / total_energy

    return {
        "sharpness_lapvar": sharpness_lapvar,
        "brightness_mean": brightness_mean,
        "contrast_std": contrast_std,
        "saturation_mean": saturation_mean,
        "noise_sigma": noise_sigma,
        "hf_energy_ratio": hf_energy_ratio,
    }

def sample_paths(paths, max_n):
    if max_n is None or max_n <= 0 or len(paths) <= max_n:
        return paths
    return random.sample(paths, max_n)

# =========================
# 실행
# =========================
rows = []
for split in SPLITS:
    split_dir = DATA_ROOT / split
    if not split_dir.exists():
        print(f"[WARN] split 폴더 없음: {split_dir}")
        continue

    paths_all = iter_images(split_dir)
    paths = sample_paths(paths_all, MAX_PER_SPLIT)

    print(f"\n[{split}] total files={len(paths_all)} | sampled={len(paths)}")
    for p in tqdm(paths, desc=f"[{split}] quality"):
        try:
            # 파일크기(압축 proxy)
            file_bytes = p.stat().st_size

            img = Image.open(p).convert("RGB")
            img_rgb = np.array(img)

            m = compute_metrics(img_rgb)
            m.update({
                "split": split,
                "path": str(p),
                "w": img.size[0],
                "h": img.size[1],
                "file_bytes": file_bytes,
                "file_kb": file_bytes / 1024.0,
            })
            rows.append(m)

        except Exception as e:
            rows.append({
                "split": split, "path": str(p),
                "error": f"{type(e).__name__}",
            })

df = pd.DataFrame(rows)

# 에러 행 제거
if "error" in df.columns:
    df_ok = df[df["error"].isna()].copy()
else:
    df_ok = df.copy()

metrics = ["sharpness_lapvar","brightness_mean","contrast_std","saturation_mean","noise_sigma","hf_energy_ratio","file_kb"]

# split별 요약 통계
summary = df_ok.groupby("split")[metrics].agg(["count","mean","std","median",
                                               lambda x: np.percentile(x, 10),
                                               lambda x: np.percentile(x, 90)])
# 컬럼 이름 정리
summary.columns = ["_".join([c[0], c[1] if isinstance(c[1], str) else "p10" if c[1].__name__=="<lambda>" else c[1]]) for c in summary.columns]
# 위 lambda가 2개라 구분이 애매하니 다시 손질
# (p10/p90 수동 계산)
p10 = df_ok.groupby("split")[metrics].quantile(0.10).add_suffix("_p10")
p90 = df_ok.groupby("split")[metrics].quantile(0.90).add_suffix("_p90")
basic = df_ok.groupby("split")[metrics].agg(["count","mean","std","median"])
basic.columns = [f"{a}_{b}" for a,b in basic.columns]
summary2 = basic.join(p10).join(p90)

# train vs test 차이(효과크기: Cohen's d)도 같이 계산
def cohens_d(a, b, eps=1e-12):
    a = np.asarray(a); b = np.asarray(b)
    na, nb = len(a), len(b)
    va, vb = a.var(ddof=1), b.var(ddof=1)
    sp = math.sqrt(((na-1)*va + (nb-1)*vb) / max(na+nb-2, 1) + eps)
    return float((a.mean() - b.mean()) / sp)

effects = []
if set(["train","test"]).issubset(set(df_ok["split"].unique())):
    for m in metrics:
        a = df_ok[df_ok["split"]=="train"][m].dropna().values
        b = df_ok[df_ok["split"]=="test"][m].dropna().values
        effects.append({"metric": m, "cohens_d_train_minus_test": cohens_d(a,b)})
effects_df = pd.DataFrame(effects)

# 저장
df_ok.to_csv(OUT_DIR/"per_image_quality.csv", index=False, encoding="utf-8-sig")
summary2.to_csv(OUT_DIR/"split_quality_summary.csv", encoding="utf-8-sig")
effects_df.to_csv(OUT_DIR/"train_vs_test_effectsize.csv", index=False, encoding="utf-8-sig")

print("\n 저장 완료:")
print(" -", OUT_DIR/"per_image_quality.csv")
print(" -", OUT_DIR/"split_quality_summary.csv")
print(" -", OUT_DIR/"train_vs_test_effectsize.csv")

print("\n=== Split summary (핵심) ===")
display(summary2)

print("\n=== Train vs Test effect size (Cohen's d; +면 train이 더 큼) ===")
display(effects_df)

# "test 품질이 더 안 좋은" 샘플 예시 (가장 블러한/가장 어두운 등)
def show_extremes(split_name, metric, n=10, ascending=True):
    sub = df_ok[df_ok["split"]==split_name][["path", metric]].dropna().sort_values(metric, ascending=ascending).head(n)
    print(f"\n[{split_name}] {metric} {'LOW' if ascending else 'HIGH'} top-{n}")
    display(sub)

# 예시 출력(원하면 지표 바꿔도 됨)
show_extremes("test", "sharpness_lapvar", n=10, ascending=True)   # 가장 blur한 test
show_extremes("test", "file_kb", n=10, ascending=True)           # 가장 작은 파일(압축 심할 가능성)
show_extremes("test", "brightness_mean", n=10, ascending=True)   # 가장 어두운 test

## sharpness 기반 class-wise resplit 생성

재분할의 핵심은 **클래스별 샘플 수는 유지하면서**,  
`sharpness_lapvar` 기준으로 train / val / test 분포를 더 가깝게 맞추는 것입니다.

In [ ]:
# ============================================
#  Class-wise pool & re-split (distribution-matched per class)
#  원본과 "클래스별 train/val/test 개수"를 그대로 유지 (=> 전체 개수도 100% 동일)
#  OUT_ROOT 누적 방지: 기존 폴더 삭제 후 재생성
# ============================================

from pathlib import Path
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import cv2
import shutil

# ----------------------------
#  설정
# ----------------------------
ROOT = Path("/content/drive/MyDrive/치아질환분류데이터셋")
SPLITS_IN = ["train", "val", "test"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

OUT_ROOT = Path("/content/drive/MyDrive/치아질환분류데이터셋_resplit_classwise_countMatched_v1")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

SEED = 42
rng = random.Random(SEED)
np.random.seed(SEED)

N_BINS_DEFAULT = 3
MIN_PER_BIN = 3
STRATIFY_METRIC = "sharpness_lapvar"

# ----------------------------
# 유틸
# ----------------------------
def iter_images(root: Path):
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            yield p

def get_class_from_path(p: Path):
    return p.parent.name  # .../<split>/<class>/<file>

def count_images_in_split(root: Path, split: str) -> int:
    return sum(1 for _ in iter_images(root / split))

def compute_sharpness(img_rgb: np.ndarray):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_32F, ksize=3)
    return float(lap.var())

def make_quantile_bins(values: np.ndarray, n_bins: int):
    v = np.asarray(values)
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return None
    n_bins = max(1, int(n_bins))
    while n_bins >= 2:
        qs = np.linspace(0, 1, n_bins + 1)
        edges = np.quantile(v, qs)
        if len(np.unique(edges)) == len(edges):
            return edges
        n_bins -= 1
    return np.array([v.min(), v.max() + 1e-6])

def assign_bin(x: float, edges: np.ndarray):
    if x >= edges[-1]:
        return len(edges) - 2
    return int(np.searchsorted(edges, x, side="right") - 1)

def stratified_split_indices(strata_labels, n_train, n_val, n_test, rng: random.Random):
    """
    strata(bin) 비율을 최대한 유지하면서,
    최종 개수 train/val/test = (n_train/n_val/n_test)를 정확히 맞춤.
    """
    N = len(strata_labels)
    idx_all = list(range(N))

    buckets = {}
    for i, s in enumerate(strata_labels):
        buckets.setdefault(s, []).append(i)

    # 1) bin별로 비율 배정(초기)
    train_idx, val_idx, test_idx = [], [], []
    for s, idxs in buckets.items():
        rng.shuffle(idxs)
        k = len(idxs)

        kt = int(round(k * (n_train / N)))
        kv = int(round(k * (n_val / N)))
        kt = min(kt, k)
        kv = min(kv, k - kt)
        ks = k - kt - kv

        train_idx += idxs[:kt]
        val_idx   += idxs[kt:kt+kv]
        test_idx  += idxs[kt+kv:kt+kv+ks]

    # 2) 전역 보정: 정확히 목표 개수로 맞춤(중복 없이)
    used = set()

    def take_unique(lst, target):
        out = []
        for i in lst:
            if i in used:
                continue
            out.append(i); used.add(i)
            if len(out) == target:
                break
        return out

    def fill_remaining(target):
        remain = [i for i in idx_all if i not in used]
        rng.shuffle(remain)
        take = remain[:target]
        for i in take:
            used.add(i)
        return take

    tr = take_unique(train_idx, n_train)
    if len(tr) < n_train:
        tr += fill_remaining(n_train - len(tr))

    va = take_unique(val_idx, n_val)
    if len(va) < n_val:
        va += fill_remaining(n_val - len(va))

    te = take_unique(test_idx, n_test)
    if len(te) < n_test:
        te += fill_remaining(n_test - len(te))

    return tr[:n_train], va[:n_val], te[:n_test]

def safe_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        dst = dst.with_name(dst.stem + f"__dup{rng.randint(0,999999)}" + dst.suffix)
    shutil.copy2(src, dst)

# ----------------------------
#  0) 원본 split 총 개수/비율 출력
# ----------------------------
orig_counts = {sp: count_images_in_split(ROOT, sp) for sp in SPLITS_IN}
orig_total = sum(orig_counts.values())
orig_ratios = {sp: orig_counts[sp] / orig_total for sp in SPLITS_IN}

print(" ORIG split counts:", orig_counts, "total=", orig_total)
print(" ORIG split ratios:", {k: round(v, 4) for k, v in orig_ratios.items()})

# ----------------------------
# 1) pool 수집 + sharpness 추출 (실패 샘플은 클래스 중앙값으로 대체)
# ----------------------------
all_paths = []
for sp in SPLITS_IN:
    sp_dir = ROOT / sp
    if sp_dir.exists():
        all_paths.extend(list(iter_images(sp_dir)))

print("Total pooled images:", len(all_paths))

rows = []
for p in tqdm(all_paths, desc="Extract sharpness"):
    cls = get_class_from_path(p)
    try:
        img = Image.open(p).convert("RGB")
        arr = np.array(img)
        sh = compute_sharpness(arr)
    except Exception:
        sh = np.nan
    rows.append({"path": str(p), "cls": cls, "sharpness_lapvar": sh})

df = pd.DataFrame(rows)

# NaN sharpness는 클래스 중앙값으로 대체(총개수 보존)
df["sharpness_lapvar"] = df.groupby("cls")["sharpness_lapvar"].transform(lambda s: s.fillna(s.median()))
df["sharpness_lapvar"] = df["sharpness_lapvar"].fillna(df["sharpness_lapvar"].median())

# 원본 split 정보(클래스별 개수 산출용)
df["orig_split"] = df["path"].apply(lambda p: p.split("/")[-3])

print("Rows with metrics:", len(df))
print("Classes:", sorted(df["cls"].unique().tolist()))

# ----------------------------
#  원본 “클래스별 split 개수” 표(이게 target이 됨)
# ----------------------------
orig_class_table = df.groupby(["cls", "orig_split"]).size().unstack(fill_value=0)
orig_class_table = orig_class_table.reindex(columns=SPLITS_IN, fill_value=0).sort_index()
print("\n===  ORIG per-class split counts (TARGET) ===")
display(orig_class_table)

# ----------------------------
# 2) 클래스별로 pool → 버킷 만들고 → "원본 클래스별 개수"로 stratified split
# ----------------------------
out_rows = []
summary_rows = []

for cls in sorted(df["cls"].unique()):
    sub = df[df["cls"] == cls].copy().reset_index(drop=True)
    N = len(sub)
    if N < 5:
        print(f"[WARN] class '{cls}' too small: N={N}. Skipping resplit.")
        continue

    # [핵심] 원본 클래스별 개수를 그대로 목표로 사용
    n_train = int(orig_class_table.loc[cls, "train"])
    n_val   = int(orig_class_table.loc[cls, "val"])
    n_test  = int(orig_class_table.loc[cls, "test"])

    # sanity
    if n_train + n_val + n_test != N:
        raise ValueError(f"[count mismatch] cls={cls} N={N} but target sum={n_train+n_val+n_test}")

    # bin 개수 자동 조정
    n_bins = N_BINS_DEFAULT
    while n_bins > 1 and (N / n_bins) < MIN_PER_BIN:
        n_bins -= 1

    edges = make_quantile_bins(sub[STRATIFY_METRIC].values, n_bins)
    if edges is None:
        edges = np.array([sub[STRATIFY_METRIC].min(), sub[STRATIFY_METRIC].max() + 1e-6])

    sub["domain_bin"] = sub[STRATIFY_METRIC].apply(lambda x: assign_bin(float(x), edges))
    strata = sub["domain_bin"].tolist()

    tr_idx, va_idx, te_idx = stratified_split_indices(strata, n_train, n_val, n_test, rng)

    sub.loc[tr_idx, "new_split"] = "train"
    sub.loc[va_idx, "new_split"] = "val"
    sub.loc[te_idx, "new_split"] = "test"

    summary_rows.append({
        "class": cls, "N": N, "bins": int(sub["domain_bin"].nunique()),
        "train_n": int((sub["new_split"]=="train").sum()),
        "val_n": int((sub["new_split"]=="val").sum()),
        "test_n": int((sub["new_split"]=="test").sum()),
    })

    out_rows.append(sub[["path", "cls", "new_split", "domain_bin"]])

df_assign = pd.concat(out_rows, axis=0).reset_index(drop=True)
df_summary = pd.DataFrame(summary_rows)

print("\n=== ✅ RESPLIT summary (per class) ===")
display(df_summary)

# ----------------------------
#  3) OUT_ROOT 누적 방지: 기존 split 폴더 삭제 후 재생성
# ----------------------------
for sp in SPLITS_IN:
    d = OUT_ROOT / sp
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)
print(" Cleared OUT_ROOT splits to prevent accumulation")

# ----------------------------
# 4) 새 폴더로 복사
# ----------------------------
print("\nCopying files...")
for split_name in SPLITS_IN:
    sub = df_assign[df_assign["new_split"] == split_name]
    for _, row in tqdm(sub.iterrows(), total=len(sub), desc=f"Copy {split_name}"):
        src = Path(row["path"])
        cls = row["cls"]
        dst = OUT_ROOT / split_name / cls / src.name
        safe_copy(src, dst)

df_assign.to_csv(OUT_ROOT / "resplit_assignments.csv", index=False, encoding="utf-8-sig")

# ----------------------------
# 5) 재분할 split 총 개수/비율 출력 (원본과 100% 동일해야 함)
# ----------------------------
out_counts = {sp: count_images_in_split(OUT_ROOT, sp) for sp in SPLITS_IN}
out_total = sum(out_counts.values())
out_ratios = {sp: out_counts[sp] / out_total for sp in SPLITS_IN}

print("\n resplit counts:", out_counts, "total=", out_total)
print(" resplit ratios:", {k: round(v, 4) for k, v in out_ratios.items()})
print(" ORIG split counts:", orig_counts, "total=", orig_total)
print(" ORIG split ratios:", {k: round(v, 4) for k, v in orig_ratios.items()})

# ----------------------------
#  6) 재분할 “클래스별 split 개수” 표 + 원본과 차이(0이어야 정상)
# ----------------------------
res_class_table = df_assign.groupby(["cls", "new_split"]).size().unstack(fill_value=0)
res_class_table = res_class_table.reindex(columns=SPLITS_IN, fill_value=0).sort_index()

print("\n===  RESPLIT per-class split counts ===")
display(res_class_table)

diff_table = orig_class_table.sub(res_class_table, fill_value=0)
print("\n===  (ORIG - RESPLIT) per-class split count difference (should be all zeros) ===")
display(diff_table)

print("\n Done. New dataset root:", OUT_ROOT)
print( Assignments CSV:", OUT_ROOT / "resplit_assignments.csv")

## 원본 vs 재분할 품질 비교 CSV 및 거리 리포트 생성

이 셀은 원본 / 재분할 데이터셋에서 다시 품질 지표를 추출하고,  
JS divergence / Wasserstein 근사 / 중앙값 비교를 통해 분포 차이를 정량화합니다.

In [ ]:
# ============================================
#  Colab 1-cell: orig + resplit(countMatched) 품질 CSV를 "항상 새로 생성" + 분포 비교 리포트 생성
#  원본/재분할 train/val/test 개수 출력 + 일치 검증(assert)
#  결과는 Drive에 저장
# ============================================


from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import cv2
import matplotlib.pyplot as plt

# ----------------------------
#  경로 설정 (중요: resplit은 countMatched 폴더로!)
# ----------------------------
ORIG_ROOT = Path("/content/drive/MyDrive/치아질환분류데이터셋")
RESPLIT_ROOT = Path("/content/drive/MyDrive/치아질환분류데이터셋_resplit_classwise_countMatched_v1")

#  Drive에 저장
WORK_DIR = Path("/content/drive/MyDrive/quality_compare_work_countMatched")
WORK_DIR.mkdir(parents=True, exist_ok=True)

ORIG_CSV = WORK_DIR / "per_image_quality_orig.csv"
RESPLIT_CSV = WORK_DIR / "per_image_quality_resplit_countMatched.csv"

REPORT_DIR = Path("/content/drive/MyDrive/quality_compare_report_countMatched")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "val", "test"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
MAX_PER_SPLIT = 0  # 0이면 전체

# ----------------------------
# 유틸
# ----------------------------
def iter_images(root: Path):
    return [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]

def robust_mad(x: np.ndarray, eps=1e-12):
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    return 1.4826 * mad + eps

def compute_metrics(img_rgb: np.ndarray):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    lap = cv2.Laplacian(gray, cv2.CV_32F, ksize=3)
    sharpness_lapvar = float(lap.var())

    brightness_mean = float(gray.mean())
    contrast_std = float(gray.std())

    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    saturation_mean = float(hsv[..., 1].mean())

    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    highpass = (gray.astype(np.float32) - blur.astype(np.float32)).flatten()
    noise_sigma = float(robust_mad(highpass))

    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    mag = cv2.magnitude(gx, gy)
    hf_energy = float(np.mean(mag))
    total_energy = float(np.mean(np.abs(gray.astype(np.float32))) + 1e-6)
    hf_energy_ratio = hf_energy / total_energy

    return {
        "sharpness_lapvar": sharpness_lapvar,
        "brightness_mean": brightness_mean,
        "contrast_std": contrast_std,
        "saturation_mean": saturation_mean,
        "noise_sigma": noise_sigma,
        "hf_energy_ratio": hf_energy_ratio,
    }

def get_class_from_path(p: Path):
    return p.parent.name  # .../<split>/<class>/<file>

def count_split(root: Path):
    counts = {}
    for sp in SPLITS:
        sp_dir = root / sp
        counts[sp] = len(iter_images(sp_dir)) if sp_dir.exists() else 0
    counts["total"] = sum(counts.values())
    return counts

def extract_quality_csv(dataset_root: Path, out_csv: Path, dataset_tag: str):
    rows = []
    for split in SPLITS:
        sp_dir = dataset_root / split
        if not sp_dir.exists():
            continue

        paths_all = iter_images(sp_dir)
        paths = paths_all if (MAX_PER_SPLIT == 0 or len(paths_all) <= MAX_PER_SPLIT) else paths_all[:MAX_PER_SPLIT]

        print(f"[{dataset_tag}/{split}] total={len(paths_all)} | using={len(paths)}")

        for p in tqdm(paths, desc=f"[{dataset_tag}/{split}] quality"):
            try:
                img = Image.open(p).convert("RGB")
                arr = np.array(img)
                m = compute_metrics(arr)
                m.update({
                    "dataset": dataset_tag,
                    "split": split,
                    "class": get_class_from_path(p),
                    "path": str(p),
                    "w": img.size[0],
                    "h": img.size[1],
                    "file_kb": p.stat().st_size / 1024.0
                })
                rows.append(m)
            except Exception:
                continue

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("✅ saved:", out_csv)
    return df

# ----------------------------
#  0) split 개수 출력 + resplit 누적(__dup) 경고 + 개수 일치 강제
# ----------------------------
orig_counts = count_split(ORIG_ROOT)
res_counts  = count_split(RESPLIT_ROOT)

print(" ORIG split counts:", orig_counts)
print(" RESPLIT split counts:", res_counts)

# __dup 파일 존재 여부(누적 흔적) 체크
dup_hits = []
for sp in SPLITS:
    sp_dir = RESPLIT_ROOT / sp
    if sp_dir.exists():
        for p in iter_images(sp_dir):
            if "__dup" in p.stem:
                dup_hits.append(str(p))
                break
if dup_hits:
    print(" WARNING: RESPLIT 폴더에 '__dup' 파일이 존재합니다. (과거 누적 가능성)")
    print("   example:", dup_hits[0])

#  핵심: countMatched라면 원본과 split 개수가 반드시 같아야 함
assert orig_counts["train"] == res_counts["train"] and orig_counts["val"] == res_counts["val"] and orig_counts["test"] == res_counts["test"], \
    f"Split counts mismatch! ORIG={orig_counts}, RESPLIT={res_counts}"

# ----------------------------
#  1) orig/resplit CSV를 '항상 새로 생성'
# ----------------------------
df_orig = extract_quality_csv(ORIG_ROOT, ORIG_CSV, "orig")
df_res  = extract_quality_csv(RESPLIT_ROOT, RESPLIT_CSV, "resplit_countMatched")

# ----------------------------
#  2) 분포거리(JS + 근사 Wasserstein) 계산
# ----------------------------
METRIC = "sharpness_lapvar"

def js_divergence(a, b, bins=50, eps=1e-12):
    a = np.asarray(a); b = np.asarray(b)
    a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
    if len(a) < 2 or len(b) < 2:
        return np.nan
    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    if lo == hi:
        return 0.0
    hist_a, _ = np.histogram(a, bins=bins, range=(lo, hi), density=True)
    hist_b, _ = np.histogram(b, bins=bins, range=(lo, hi), density=True)
    p = hist_a + eps; q = hist_b + eps
    p = p / p.sum(); q = q / q.sum()
    m = 0.5 * (p + q)
    js = 0.5 * (np.sum(p * np.log(p / m)) + np.sum(q * np.log(q / m)))
    return float(js)

def wasserstein_approx(a, b):
    a = np.asarray(a); b = np.asarray(b)
    a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
    if len(a) < 2 or len(b) < 2:
        return np.nan
    n = min(len(a), len(b), 2000)
    if len(a) > n: a = np.random.choice(a, size=n, replace=False)
    if len(b) > n: b = np.random.choice(b, size=n, replace=False)
    a = np.sort(a); b = np.sort(b)
    return float(np.mean(np.abs(a - b)))

classes = sorted(set(df_orig["class"].unique()).intersection(set(df_res["class"].unique())))
rows = []
for cls in classes:
    o_tr = df_orig[(df_orig["split"]=="train") & (df_orig["class"]==cls)][METRIC].values
    o_te = df_orig[(df_orig["split"]=="test")  & (df_orig["class"]==cls)][METRIC].values
    r_tr = df_res[(df_res["split"]=="train") & (df_res["class"]==cls)][METRIC].values
    r_te = df_res[(df_res["split"]=="test")  & (df_res["class"]==cls)][METRIC].values

    rows.append({
        "class": cls,
        "orig_js_train_vs_test": js_divergence(o_tr, o_te),
        "resplit_js_train_vs_test": js_divergence(r_tr, r_te),
        "orig_wass_train_vs_test": wasserstein_approx(o_tr, o_te),
        "resplit_wass_train_vs_test": wasserstein_approx(r_tr, r_te),
        "orig_train_median": float(np.nanmedian(o_tr)) if len(o_tr)>0 else np.nan,
        "orig_test_median":  float(np.nanmedian(o_te)) if len(o_te)>0 else np.nan,
        "res_train_median":  float(np.nanmedian(r_tr)) if len(r_tr)>0 else np.nan,
        "res_test_median":   float(np.nanmedian(r_te)) if len(r_te)>0 else np.nan,
    })

df_dist = pd.DataFrame(rows)
df_dist["js_drop"] = df_dist["orig_js_train_vs_test"] - df_dist["resplit_js_train_vs_test"]
df_dist["wass_drop"] = df_dist["orig_wass_train_vs_test"] - df_dist["resplit_wass_train_vs_test"]
df_dist = df_dist.sort_values("orig_js_train_vs_test", ascending=False)

dist_csv = REPORT_DIR / "classwise_train_vs_test_distance.csv"
df_dist.to_csv(dist_csv, index=False, encoding="utf-8-sig")
print("\n✅ saved:", dist_csv)
display(df_dist)

# ----------------------------
#  3) 분포 그래프 저장
# ----------------------------
plot_classes = ["ulcers", "discoloration"] + df_dist["class"].head(2).tolist()
plot_classes = [c for c in dict.fromkeys(plot_classes) if c in classes]

def plot_hist(df, dataset_name, cls, split_a="train", split_b="test", bins=50):
    a = df[(df["split"]==split_a) & (df["class"]==cls)][METRIC].values
    b = df[(df["split"]==split_b) & (df["class"]==cls)][METRIC].values
    a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
    if len(a) < 2 or len(b) < 2:
        return
    lo = min(a.min(), b.min()); hi = max(a.max(), b.max())
    if lo == hi:
        return
    plt.figure(figsize=(7,4))
    plt.hist(a, bins=bins, range=(lo,hi), alpha=0.6, density=True, label=split_a)
    plt.hist(b, bins=bins, range=(lo,hi), alpha=0.6, density=True, label=split_b)
    plt.title(f"{dataset_name} | {cls} | {METRIC}")
    plt.xlabel(METRIC); plt.ylabel("density")
    plt.legend()
    plt.tight_layout()
    out = REPORT_DIR / f"{dataset_name}_{cls}_{METRIC}_{split_a}_vs_{split_b}.png"
    plt.savefig(out, dpi=160)
    plt.close()

for cls in plot_classes:
    plot_hist(df_orig, "orig", cls)
    plot_hist(df_res,  "resplit_countMatched", cls)

print("\n plots saved under:", REPORT_DIR)
print(" orig csv:", ORIG_CSV)
print(" resplit csv:", RESPLIT_CSV)

In [ ]:
from IPython.display import Image as IPyImage, display
from pathlib import Path

REPORT_DIR = Path("/content/drive/MyDrive/quality_compare_report_countMatched")
if REPORT_DIR.exists():
    png_files = sorted(REPORT_DIR.glob("*.png"))
    print("report dir:", REPORT_DIR)
    print("num png files:", len(png_files))
    for img_path in png_files:
        print("\n---", img_path.name, "---")
        display(IPyImage(filename=str(img_path)))
else:
    print("report dir not found:", REPORT_DIR)

# E. 재분할 데이터셋 기준 모델 학습 결과

이 섹션은 `sharpness` 기준으로 다시 구축한 `countMatched` 데이터셋을 사용해  
동일한 3개 CNN 백본을 다시 평가하는 단계입니다.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision.models import VGG19_BN_Weights, ResNet50_Weights, DenseNet121_Weights
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler

import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
#  재분할 데이터셋만 사용
# ============================================================
from pathlib import Path

#  올바른 재분할 폴더(개수 100% 매칭)
DATA_ROOT = "/content/drive/MyDrive/치아질환분류데이터셋_resplit_classwise_countMatched_v1"
#DATA_ROOT= '/content/drive/MyDrive/치아질환분류데이터셋_resplit_ROI_20260302-093923' #세그모델로 roi crop한거
#  원본과 split 개수가 같은지 강제 체크
ORIG_ROOT = "/content/drive/MyDrive/치아질환분류데이터셋"

def count_imgs(d):
    exts = {".jpg",".jpeg",".png",".bmp",".webp"}
    d = Path(d)
    return sum(1 for p in d.rglob("*") if p.is_file() and p.suffix.lower() in exts)

orig_counts = {sp: count_imgs(f"{ORIG_ROOT}/{sp}") for sp in ["train","val","test"]}
res_counts  = {sp: count_imgs(f"{DATA_ROOT}/{sp}") for sp in ["train","val","test"]}

print(" ORIG counts:", orig_counts)
print(" RESPLIT(countMatched) counts:", res_counts)

assert orig_counts == res_counts, f"Split counts mismatch! ORIG={orig_counts}, RESPLIT={res_counts}"

# ============================================================
#  Drive 저장 폴더(재분할 전용)
# ============================================================
OUT_DIR = "/content/drive/MyDrive/model_compare_bestft_fixedclass/resplit_sharpness"
os.makedirs(OUT_DIR, exist_ok=True)
print(" OUT_DIR =", OUT_DIR)

# ============================================================
#  클래스 순서 고정
# ============================================================
FIXED_CLASSES = ["calculus", "caries", "discoloration", "hypodontia", "ulcers"]

class RemapTargetsDataset(torch.utils.data.Dataset):
    def __init__(self, base_ds, fixed_classes):
        self.base = base_ds
        self.fixed_classes = list(fixed_classes)
        self.fixed_class_to_idx = {c:i for i,c in enumerate(self.fixed_classes)}

        ds_set = set(base_ds.classes)
        fixed_set = set(self.fixed_classes)
        if ds_set != fixed_set:
            missing = sorted(list(fixed_set - ds_set))
            extra = sorted(list(ds_set - fixed_set))
            raise ValueError(f"[class mismatch] missing={missing}, extra={extra}. base_ds.classes={base_ds.classes}")

        inv_old = {old_idx: cls for cls, old_idx in base_ds.class_to_idx.items()}
        self.old_to_fixed = {old_idx: self.fixed_class_to_idx[inv_old[old_idx]] for old_idx in inv_old}

        self.classes = self.fixed_classes
        self.class_to_idx = self.fixed_class_to_idx
        self.targets = [self.old_to_fixed[int(t)] for t in base_ds.targets]

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        x, y_old = self.base[idx]
        y = self.old_to_fixed[int(y_old)]
        return x, y

# ============================================================
# transforms
# ============================================================
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.0),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

eval_transforms = transforms.Compose([
    transforms.Resize(size=(256)),
    transforms.CenterCrop(size=(224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# ============================================================
# dataset 로드 (raw -> fixed mapping)
# ============================================================
train_ds_raw = datasets.ImageFolder(f"{DATA_ROOT}/train", transform=train_transforms)
val_ds_raw   = datasets.ImageFolder(f"{DATA_ROOT}/val",   transform=eval_transforms)
test_ds_raw  = datasets.ImageFolder(f"{DATA_ROOT}/test",  transform=eval_transforms)

train_ds = RemapTargetsDataset(train_ds_raw, FIXED_CLASSES)
val_ds   = RemapTargetsDataset(val_ds_raw, FIXED_CLASSES)
test_ds  = RemapTargetsDataset(test_ds_raw, FIXED_CLASSES)

class_names = train_ds.classes
num_classes = len(class_names)

print("DATA_ROOT:", DATA_ROOT)
print("FIXED class order:", class_names)
print("fixed class_to_idx:", train_ds.class_to_idx)

# ============================================================
# sampler 유지
# ============================================================
labels = torch.tensor(train_ds.targets, dtype=torch.long)
class_count = torch.bincount(labels, minlength=num_classes).float().clamp_min(1.0)
alpha = 1.0
class_weight = (1.0 / class_count) ** alpha
sample_weight = class_weight[labels]

SEED = 42
g = torch.Generator(); g.manual_seed(SEED)

train_sampler = WeightedRandomSampler(
    weights=sample_weight,
    num_samples=len(sample_weight),
    replacement=True,
    generator=g
)

train_loader = DataLoader(train_ds, batch_size=32, sampler=train_sampler, shuffle=False, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

# ============================================================
# 모델별 fine-tuning 정책 (VGG 기존 / ResNet,DenseNet best-ft)
# ============================================================
def freeze_all(net):
    for p in net.parameters():
        p.requires_grad = False

def enable_bn_affine(net):
    for m in net.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            if m.weight is not None: m.weight.requires_grad = True
            if m.bias is not None:   m.bias.requires_grad = True

def build_vgg19_bn(num_classes):
    net = models.vgg19_bn(weights=VGG19_BN_Weights.DEFAULT)
    freeze_all(net)
    enable_bn_affine(net)
    net.classifier[6] = nn.Linear(4096, num_classes)
    for p in net.classifier[6].parameters():
        p.requires_grad = True
    return net

def build_resnet50_bestft(num_classes):
    net = models.resnet50(weights=ResNet50_Weights.DEFAULT)
    freeze_all(net)
    for p in net.layer4.parameters():
        p.requires_grad = True
    enable_bn_affine(net)
    net.fc = nn.Linear(net.fc.in_features, num_classes)
    for p in net.fc.parameters():
        p.requires_grad = True
    return net

def build_densenet121_bestft(num_classes):
    net = models.densenet121(weights=DenseNet121_Weights.DEFAULT)
    freeze_all(net)
    for p in net.features.denseblock4.parameters():
        p.requires_grad = True
    if hasattr(net.features, "norm5"):
        for p in net.features.norm5.parameters():
            p.requires_grad = True
    enable_bn_affine(net)
    net.classifier = nn.Linear(net.classifier.in_features, num_classes)
    for p in net.classifier.parameters():
        p.requires_grad = True
    return net

def make_sgd_param_groups(net, head_keys, backbone_keys,
                          lr_head=1e-3, lr_backbone=1e-4, lr_bn=3e-4,
                          weight_decay=5e-4, momentum=0.9):
    head_params, backbone_params, bn_params = [], [], []
    for name, p in net.named_parameters():
        if not p.requires_grad:
            continue
        if "bn" in name.lower() or "batchnorm" in name.lower():
            bn_params.append(p)
            continue
        if any(k in name for k in head_keys):
            head_params.append(p)
        elif any(k in name for k in backbone_keys):
            backbone_params.append(p)
        else:
            backbone_params.append(p)

    param_groups = []
    if head_params:
        param_groups.append({"params": head_params, "lr": lr_head, "weight_decay": weight_decay, "momentum": momentum})
    if backbone_params:
        param_groups.append({"params": backbone_params, "lr": lr_backbone, "weight_decay": weight_decay, "momentum": momentum})
    if bn_params:
        param_groups.append({"params": bn_params, "lr": lr_bn, "weight_decay": 0.0, "momentum": momentum})

    return optim.SGD(param_groups, lr=lr_head, momentum=momentum, weight_decay=weight_decay)

# ============================================================
# TEST 평가
# ============================================================
@torch.no_grad()
def evaluate_on_loader(net, loader, criterion, num_classes, device):
    net.eval()
    cm = torch.zeros((num_classes, num_classes), dtype=torch.int64)
    total_loss, total_n = 0.0, 0

    for x, y in loader:
        x = x.to(device); y = y.to(device)
        logits = net(x)
        loss = criterion(logits, y)
        bs = y.size(0)
        total_loss += float(loss.item()) * bs
        total_n += bs

        pred = logits.argmax(dim=1)
        for t, p in zip(y.view(-1), pred.view(-1)):
            cm[t.long(), p.long()] += 1

    cm_f = cm.float()
    tp = torch.diag(cm_f)
    fn = cm_f.sum(dim=1) - tp
    fp = cm_f.sum(dim=0) - tp

    eps = 1e-12
    recall = tp / (tp + fn + eps)
    precision = tp / (tp + fp + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)

    macro_f1 = float(f1.mean().item())
    balanced_acc = float(recall.mean().item())
    acc = float(tp.sum().item() / max(cm_f.sum().item(), 1.0))
    avg_loss = total_loss / max(total_n, 1)

    return cm.cpu().numpy(), recall.cpu().numpy(), acc, macro_f1, balanced_acc, avg_loss

def normalize_rows(cm):
    row_sum = cm.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1
    return cm / row_sum

criterion = nn.CrossEntropyLoss()

# ============================================================
# 학습 + 리포트 생성 (resplit 전용)
# ============================================================
MODELS = [
    ("vgg19_bn", build_vgg19_bn),
    ("resnet50", build_resnet50_bestft),
    ("densenet121", build_densenet121_bestft),
]

results = []

for name, builder in MODELS:
    print("\n" + "="*80)
    print(f"Training: {name} (resplit)")
    print("="*80)

    net = builder(num_classes).to(device)

    if name == "vgg19_bn":
        optimizer = None
    elif name == "resnet50":
        optimizer = make_sgd_param_groups(net, head_keys=["fc."], backbone_keys=["layer4."],
                                          lr_head=1e-3, lr_backbone=1e-4, lr_bn=3e-4)
    else:
        optimizer = make_sgd_param_groups(net, head_keys=["classifier."],
                                          backbone_keys=["features.denseblock4.", "features.norm5."],
                                          lr_head=1e-3, lr_backbone=1e-4, lr_bn=3e-4)

    ckpt_path = os.path.join(OUT_DIR, f"best_{name}_resplit_sharpness.pt")

    train_model_v2(
        optimizer_name="SGD",
        net=net,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        num_epochs=150,
        device=device,
        ckpt_path=ckpt_path,
        use_cutmix=False,
        use_mixup=False,
        optimizer=optimizer,
        default_lr=3e-5
    )

    ckpt = torch.load(ckpt_path, map_location=device)
    if "model_state_dict" in ckpt:
        net.load_state_dict(ckpt["model_state_dict"], strict=True)
    else:
        net.load_state_dict(ckpt["model"], strict=True)

    cm, recall, acc, macro_f1, bal_acc, loss = evaluate_on_loader(net, test_loader, criterion, num_classes, device)
    cm_norm = normalize_rows(cm)

    recall_str = " | ".join([f"{cls}:{recall[i]:.3f}" for i, cls in enumerate(class_names)])
    print(f"[{name}/resplit] TEST loss={loss:.4f} | acc={acc*100:.2f}% | macroF1={macro_f1:.4f} | bal_acc={bal_acc*100:.2f}%")
    print(f"[{name}/resplit] TEST Per-class Recall: {recall_str}")

    results.append({
        "model": name,
        "cm_norm": cm_norm,
        "recall": recall,
        "acc": acc,
        "macro_f1": macro_f1,
        "bal_acc": bal_acc,
        "loss": loss
    })

# 리포트 저장(덮어쓰기 방지: resplit 태그)
table_cols = ["Class"] + [r["model"] for r in results]
table_data = []
for i, cls in enumerate(class_names):
    table_data.append([cls] + [f"{r['recall'][i]:.3f}" for r in results])

overall_cols = ["Metric"] + [r["model"] for r in results]
overall_data = [
    ["Test Loss"] + [f"{r['loss']:.4f}" for r in results],
    ["Test Acc"] + [f"{r['acc']*100:.2f}%" for r in results],
    ["Test Macro-F1"] + [f"{r['macro_f1']:.4f}" for r in results],
    ["Test Balanced Acc"] + [f"{r['bal_acc']*100:.2f}%" for r in results],
]

fig = plt.figure(figsize=(18, 11))
for idx, r in enumerate(results):
    ax = fig.add_subplot(2, 3, idx+1)
    im = ax.imshow(r["cm_norm"], aspect="auto")
    ax.set_title(f"{r['model']} (resplit) normalized CM")
    ax.set_xlabel("Pred"); ax.set_ylabel("True")
    ax.set_xticks(range(num_classes)); ax.set_yticks(range(num_classes))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)

    for i in range(num_classes):
        for j in range(num_classes):
            ax.text(j, i, f"{r['cm_norm'][i,j]:.2f}", ha="center", va="center", fontsize=7)

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax_tbl = fig.add_subplot(2, 1, 2)
ax_tbl.axis("off")

tbl1 = ax_tbl.table(cellText=table_data, colLabels=table_cols, loc="upper center")
tbl1.auto_set_font_size(False); tbl1.set_fontsize(10); tbl1.scale(1.0, 1.25)

tbl2 = ax_tbl.table(cellText=overall_data, colLabels=overall_cols, loc="lower center")
tbl2.auto_set_font_size(False); tbl2.set_fontsize(11); tbl2.scale(1.0, 1.35)

ax_tbl.set_title("TEST Report (resplit): Per-class Recall + Overall Metrics", pad=10)

plt.tight_layout()
pdf_path = os.path.join(OUT_DIR, "report_onepage_resplit_sharpness.pdf")
png_path = os.path.join(OUT_DIR, "report_onepage_resplit_sharpness.png")
plt.savefig(pdf_path)
plt.savefig(png_path, dpi=220)
plt.close()

print("\n Saved report to Drive:")
print(" -", pdf_path)
print(" -", png_path)
print("Saved ckpts to Drive inside:", OUT_DIR)

In [ ]:
# =========================
# Grad-CAM 시각화: 원본 + 모델별 Grad-CAM 비교 그리드
# =========================
from pathlib import Path
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image
import matplotlib.pyplot as plt

from torchvision import transforms
from torchvision.datasets import ImageFolder
import torchvision.models as models

# =========================================
# 0) Config
# =========================================
DATA_ROOT = "/content/drive/MyDrive/치아질환분류데이터셋_resplit_classwise_countMatched_v1"
TEST_DIR = f"{DATA_ROOT}/test"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)

MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

IMG_SIZE = 224
SAVE_DIR = "/content/drive/MyDrive/model_compare_bestft_fixedclass/resplit_sharpness/gradcam_compare"
os.makedirs(SAVE_DIR, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================
# 1) Dataset
# =========================================
test_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

test_ds = ImageFolder(TEST_DIR, transform=test_tf)
class_to_idx = test_ds.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}

def find_discoloration_idx(class_to_idx: dict):
    for k, v in class_to_idx.items():
        if k.lower() == "discoloration":
            return v, k
    for k, v in class_to_idx.items():
        if "discolor" in k.lower():
            return v, k
    raise ValueError(f"Cannot find discoloration class in: {list(class_to_idx.keys())}")

DIS_IDX, DIS_NAME = find_discoloration_idx(class_to_idx)
print(" Discoloration class:", DIS_NAME, "->", DIS_IDX)

dis_samples = [(i, p, y) for i, (p, y) in enumerate(test_ds.samples) if y == DIS_IDX]
print(" Discoloration test samples:", len(dis_samples))
assert len(dis_samples) > 0, "test에 discoloration 샘플이 0개입니다."

# =========================================
# 2) Grad-CAM
# =========================================
def find_last_conv_layer(model: nn.Module):
    last_name, last_module = None, None
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            last_name, last_module = name, module
    if last_module is None:
        raise ValueError("No Conv2d layer found.")
    return last_name, last_module

class GradCAM:
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None

        def fwd_hook(module, inp, out):
            self.activations = out

        def bwd_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0]

        self.h1 = self.target_layer.register_forward_hook(fwd_hook)
        self.h2 = self.target_layer.register_full_backward_hook(bwd_hook)

    def remove_hooks(self):
        self.h1.remove()
        self.h2.remove()

    @torch.enable_grad()
    def generate(self, x: torch.Tensor, target_class: int):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        score = logits[:, target_class].sum()
        score.backward(retain_graph=True)

        A = self.activations
        dYdA = self.gradients

        weights = dYdA.mean(dim=(2,3), keepdim=True)
        cam = (weights * A).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(x.shape[2], x.shape[3]), mode="bilinear", align_corners=False)

        cam = cam.squeeze().detach()
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-12)
        return cam, logits.detach()

def unnormalize(img_t: torch.Tensor, mean=MEAN, std=STD):
    mean = torch.tensor(mean, device=img_t.device).view(3,1,1)
    std = torch.tensor(std, device=img_t.device).view(3,1,1)
    x = img_t * std + mean
    x = x.clamp(0,1)
    return x.permute(1,2,0).cpu().numpy()

def overlay_cam(rgb01: np.ndarray, cam01: np.ndarray, alpha=0.45):
    cmap = plt.get_cmap("jet")
    heat = cmap(cam01)[...,:3]
    out = (1 - alpha) * rgb01 + alpha * heat
    return np.clip(out, 0, 1)

# =========================================
# 3) Model build/load
# =========================================
def build_model(arch: str, num_classes: int):
    arch = arch.lower()
    if arch == "vgg19_bn":
        m = models.vgg19_bn(weights=None)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
        return m
    if arch == "resnet50":
        m = models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m
    if arch == "densenet121":
        m = models.densenet121(weights=None)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
        return m
    raise ValueError(f"Unknown arch: {arch}")

def load_checkpoint(model: nn.Module, ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        sd = ckpt["model_state_dict"]
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        sd = ckpt["state_dict"]
    elif isinstance(ckpt, dict):
        sd = ckpt
    else:
        raise ValueError("Unsupported checkpoint format.")

    new_sd = {k.replace("module.",""): v for k, v in sd.items()}
    model.load_state_dict(new_sd, strict=True)
    return model

MODEL_SPECS = [
    {"name": "VGG19_BN", "arch": "vgg19_bn", "ckpt": "/content/drive/MyDrive/model_compare_bestft_fixedclass/resplit_sharpness/best_vgg19_bn_resplit_sharpness.pt"},
    {"name": "ResNet50", "arch": "resnet50", "ckpt": "/content/drive/MyDrive/model_compare_bestft_fixedclass/resplit_sharpness/best_resnet50_resplit_sharpness.pt"},
    {"name": "DenseNet121", "arch": "densenet121", "ckpt": "/content/drive/MyDrive/model_compare_bestft_fixedclass/resplit_sharpness/best_densenet121_resplit_sharpness.pt"},
]

NUM_CLASSES = len(class_to_idx)

#  모델들 미리 로드 + 각 모델별 Grad-CAM 준비
models_dict = {}
for spec in MODEL_SPECS:
    m = build_model(spec["arch"], NUM_CLASSES)
    m = load_checkpoint(m, spec["ckpt"])
    m = m.to(device).eval()

    layer_name, layer_module = find_last_conv_layer(m)
    print(f"✅ {spec['name']} target layer:", layer_name)
    cam_engine = GradCAM(m, layer_module)

    models_dict[spec["name"]] = {"model": m, "cam": cam_engine}

# =========================================
# 4) 한 화면 비교 시각화 (rows=images, cols=원본+모델들)
# =========================================
alpha = 0.45
target_class = DIS_IDX

n_imgs = len(dis_samples)          # 18 예상
n_models = len(MODEL_SPECS)
n_cols = n_models + 1              #  원본 컬럼(1개) 추가

#  figure 가로폭도 컬럼 수에 맞춰 확장
fig = plt.figure(figsize=(n_cols * 4.6, n_imgs * 2.6))

for r, (idx, img_path, y) in enumerate(dis_samples):
    x, _ = test_ds[idx]
    x1 = x.unsqueeze(0).to(device)
    rgb01 = unnormalize(x.to(device))

    base = os.path.splitext(os.path.basename(img_path))[0]

    # -------------------------
    # (A)  원본 이미지 컬럼
    # -------------------------
    ax0 = plt.subplot(n_imgs, n_cols, r * n_cols + 1)   # col=0
    ax0.imshow(rgb01)
    ax0.axis("off")
    if r == 0:
        ax0.set_title("Original", fontsize=10)          #  첫 줄 헤더

    #  이미지 id 표시는 원본 칸에 표시(가장 직관적)
    ax0.text(3, 12, base, color="white", fontsize=8,
             bbox=dict(facecolor="black", alpha=0.55, pad=2))

    # -------------------------
    # (B) 모델별 Grad-CAM 컬럼들
    # -------------------------
    for m_i, spec in enumerate(MODEL_SPECS):
        name = spec["name"]
        m = models_dict[name]["model"]
        cam_engine = models_dict[name]["cam"]

        with torch.no_grad():
            logits0 = m(x1)
            prob = F.softmax(logits0, dim=1)[0]
            pred = int(torch.argmax(prob).item())
            pred_name = idx_to_class[pred]
            pred_p = float(prob[pred].item())
            dis_p = float(prob[DIS_IDX].item())
            correct = (pred == DIS_IDX)

        cam01, _ = cam_engine.generate(x1, target_class=target_class)
        over = overlay_cam(rgb01, cam01.cpu().numpy(), alpha=alpha)

        #  col index: 원본(0) 다음이 첫 모델(1)
        col = 1 + m_i
        ax = plt.subplot(n_imgs, n_cols, r * n_cols + col + 1)
        ax.imshow(over)
        ax.axis("off")

        # 첫 줄(row=0)에만 모델 이름 헤더처럼
        if r == 0:
            ax.set_title(f"{name}\nPred={pred_name}({pred_p:.2f}) P(dis)={dis_p:.2f}", fontsize=9)
        else:
            ax.set_title(f"Pred={pred_name}({pred_p:.2f}) P(dis)={dis_p:.2f} | ok={correct}", fontsize=8)

grid_path = os.path.join(SAVE_DIR, "GRID__ORIG_plus_compare_VGG_ResNet_DenseNet__target_discoloration.png")  # ✅ [변경] 파일명
plt.tight_layout()
plt.savefig(grid_path, dpi=200)
plt.show()

# hook 해제
for name in models_dict:
    models_dict[name]["cam"].remove_hooks()

print("✅ saved:", grid_path)

In [ ]:
from IPython.display import Image as IPyImage, display, IFrame
from pathlib import Path

RESPLIT_OUT_DIR = Path("/content/drive/MyDrive/model_compare_bestft_fixedclass/resplit_sharpness")
png_path = RESPLIT_OUT_DIR / "report_onepage_resplit_sharpness.png"
pdf_path = RESPLIT_OUT_DIR / "report_onepage_resplit_sharpness.pdf"

print("PNG exists?", png_path.exists(), png_path)
print("PDF exists?", pdf_path.exists(), pdf_path)

if png_path.exists():
    display(IPyImage(filename=str(png_path)))
if pdf_path.exists():
    display(IFrame(src=str(pdf_path), width=1100, height=750))

# F. segmentation ROI 확장

재분할만으로도 입력 분포 문제는 상당 부분 완화되지만,  
여전히 구강 외 주변 문맥이 많으면 질환 단서보다 배경을 학습할 위험이 있습니다.

이 확장 실험에서는 segmentation 기반 ROI를 이용해  
**치아 + 잇몸 주변**만 더 집중적으로 보도록 데이터를 다시 구성합니다.

## segmentation 모델 설치

In [ ]:
!pip install -q segmentation_models_pytorch

## segmentation ROI 데이터셋 생성

이 셀은 FPN + EfficientNet-b7 기반 치아 segmentation 결과를 이용해  
재분할 데이터셋에서 이미지의 도메인 시프트를 줄이고, ROI crop 버전을 새로 생성합니다.

In [ ]:
import os, time, random
from pathlib import Path
import numpy as np
import cv2
from PIL import Image
from tqdm.auto import tqdm
import torch
import segmentation_models_pytorch as smp
from torchvision import transforms as T
import matplotlib.pyplot as plt

# =========================
# 0) 설정 (여기만 바꾸면 됨)
# =========================
SRC_ROOT = "/content/drive/MyDrive/치아질환분류데이터셋_resplit_classwise_countMatched_v1"  # 재분할된 원본 데이터셋 루트
BASE_DST_ROOT = "/content/drive/MyDrive/치아질환분류데이터셋_resplit_ROI"  # 새 ROI 데이터셋 기본명
WEIGHT = "/content/drive/MyDrive/NIA/toothnumber/utils/tooth_number_saved_weight.pt"

# ROI 파라미터(치아+잇몸/입술 근처 포함)
IM_SIZE = 224
DILATE_K = 71          #  입술/잇몸 문맥 포함하려고 dilate 더 크게(권장: 61~91 탐색)
DILATE_IT = 1

BBOX_MARGIN = 20       #  좌우 기본 여백
MIN_ROI_PIXELS = 2000

MASK_INSIDE_CROP = False  #  픽셀단위로 깎아 검정조각 만드는 걸 방지(자연스러운 crop 유지)

#  비대칭 확장(코 밑/입술 포함 목적)
EXPAND_LEFT = 60
EXPAND_RIGHT = 60
EXPAND_TOP = 180       #   위쪽(코 밑 방향)을 더 크게
EXPAND_BOTTOM = 140    #  아래쪽(아랫입술/잇몸)도 크게

# 샘플 확인 개수
N_SHOW = 5
random.seed(42)

# =========================
# 1) DST_ROOT를 "항상 새로" 만들기 (누적/덮어쓰기 방지)
# =========================
stamp = time.strftime("%Y%m%d-%H%M%S")
DST_ROOT = f"{BASE_DST_ROOT}_{stamp}"
Path(DST_ROOT).mkdir(parents=True, exist_ok=False)
print("SRC_ROOT:", SRC_ROOT)
print("DST_ROOT:", DST_ROOT)

# =========================
# 2) 모델 로드
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = smp.FPN("efficientnet-b7", encoder_weights=None, classes=33, activation=None)
state = torch.load(WEIGHT, map_location=device)
model.load_state_dict(state, strict=True)
model.to(device).eval()

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
tfm = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])

@torch.no_grad()
def tooth_labelmap(pil_img: Image.Image):
    ow, oh = pil_img.size
    x_img = pil_img.resize((IM_SIZE, IM_SIZE), Image.BILINEAR)
    x = tfm(x_img).unsqueeze(0).to(device)
    logits = model(x)
    pred = torch.argmax(logits, dim=1)[0].cpu().numpy().astype(np.uint8)  # (224,224)
    pred = cv2.resize(pred, (ow, oh), interpolation=cv2.INTER_NEAREST)
    return pred

def dilate(mask_bool, k=DILATE_K, it=DILATE_IT):
    ker = np.ones((k, k), np.uint8)
    return cv2.dilate(mask_bool.astype(np.uint8), ker, iterations=it).astype(bool)

def bbox(mask_bool):
    ys, xs = np.where(mask_bool)
    if len(xs) == 0:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())

def expand_asym(x1, y1, x2, y2, w, h,
                left=60, right=60, top=180, bottom=140):
    """
     bbox 비대칭 확장: 입술/잇몸/코 밑 문맥을 보존하기 위함
    """
    x1 = max(0, x1 - left)
    y1 = max(0, y1 - top)
    x2 = min(w - 1, x2 + right)
    y2 = min(h - 1, y2 + bottom)
    return x1, y1, x2, y2

def save_image(arr, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(arr).save(out_path)

# =========================
# 3) ROI 데이터셋 생성 (원본 절대 변경 X) + tqdm 진행바
# =========================
exts = {".jpg", ".jpeg", ".png"}
splits = ["train", "val", "test"]

# split별 파일 리스트를 미리 수집해서 tqdm total로 사용
split_files = {}
for sp in splits:
    sp_dir = Path(SRC_ROOT) / sp
    if not sp_dir.exists():
        continue
    files = [p for p in sp_dir.rglob("*") if p.suffix.lower() in exts]
    split_files[sp] = files

#  통계(디버깅/레포트에 유용)
stats = {sp: {"total": 0, "saved_original": 0, "cropped": 0, "no_bbox": 0} for sp in splits}

for sp in splits:
    if sp not in split_files:
        print(f"[{sp}] not found. skip.")
        continue

    files = split_files[sp]
    pbar = tqdm(files, desc=f"Building ROI dataset [{sp}]", unit="img")

    for img_path in pbar:
        stats[sp]["total"] += 1

        rel = img_path.relative_to(Path(SRC_ROOT))   # train/class/xxx.png
        out_path = Path(DST_ROOT) / rel              # DST_ROOT/train/class/xxx.png

        pil = Image.open(img_path).convert("RGB")
        rgb = np.array(pil)
        h, w = rgb.shape[:2]

        tmap = tooth_labelmap(pil)
        tooth_union = (tmap > 0)
        roi = dilate(tooth_union)

        # ROI 너무 작으면 -> 새 데이터셋에는 원본 그대로 저장(치아가 거의 없는 ulcers 등 방어)
        if int(roi.sum()) < MIN_ROI_PIXELS:
            save_image(rgb, out_path)
            stats[sp]["saved_original"] += 1
            continue

        bb = bbox(roi)
        if bb is None:
            save_image(rgb, out_path)
            stats[sp]["no_bbox"] += 1
            continue

        #  대칭 확장 대신
        # x1,y1,x2,y2 = expand(*bb, m=BBOX_MARGIN, w=w, h=h)

        #  비대칭 확장 + 좌우/상하 확장량 반영
        x1, y1, x2, y2 = expand_asym(
            *bb, w=w, h=h,
            left=EXPAND_LEFT, right=EXPAND_RIGHT,
            top=EXPAND_TOP, bottom=EXPAND_BOTTOM
        )

        #  자연스러운 crop(검정 조각 방지)
        crop = rgb[y1:y2+1, x1:x2+1].copy()

        #  ROI 밖을 검정 처리 -> 조각난 형태 유발
        # if MASK_INSIDE_CROP:
        #     roi_crop = roi[y1:y2+1, x1:x2+1]
        #     crop[~roi_crop] = 0

        save_image(crop, out_path)
        stats[sp]["cropped"] += 1

        # 진행바 postfix로 상태 표시(속도/진행 외, 정책이 잘 먹는지)
        pbar.set_postfix({
            "cropped": stats[sp]["cropped"],
            "orig": stats[sp]["saved_original"]
        })

print("ROI dataset created (original untouched):", DST_ROOT)
print("stats:", stats)

# =========================
# 4) split별/클래스별 5장 샘플 확인
# =========================
def list_classes(split_root: Path):
    classes = [p.name for p in split_root.iterdir() if p.is_dir()]
    classes.sort()
    return classes

def sample_images(class_dir: Path, n=5):
    imgs = [p for p in class_dir.rglob("*") if p.suffix.lower() in exts]
    imgs.sort()
    if len(imgs) == 0:
        return []
    if len(imgs) <= n:
        return imgs
    return random.sample(imgs, n)

def show_grid(title, paths, cols=5):
    if len(paths) == 0:
        print(title, "=> (no images)")
        return
    rows = int(np.ceil(len(paths) / cols))
    plt.figure(figsize=(cols * 3, rows * 3))
    plt.suptitle(title)
    for i, p in enumerate(paths):
        img = Image.open(p).convert("RGB")
        plt.subplot(rows, cols, i+1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(p.name, fontsize=9)
    plt.tight_layout()
    plt.show()

for sp in splits:
    sp_root = Path(DST_ROOT) / sp
    if not sp_root.exists():
        continue
    classes = list_classes(sp_root)
    print(f"\n===== [{sp}] classes: {len(classes)} =====")
    for cls in classes:
        picks = sample_images(sp_root / cls, n=N_SHOW)
        show_grid(f"{sp} / {cls} (n={len(picks)})", picks, cols=N_SHOW)

In [ ]:
from pathlib import Path
import random
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

# 이전 셀에서 생성된 SRC_ROOT / DST_ROOT 재사용
exts = {".jpg", ".jpeg", ".png"}
splits = ["train", "val", "test"]
N_SHOW = 5
random.seed(42)

def list_classes(split_root: Path):
    classes = [p.name for p in split_root.iterdir() if p.is_dir()]
    classes.sort()
    return classes

def sample_images(class_dir: Path, n=5):
    imgs = [p for p in class_dir.rglob("*") if p.suffix.lower() in exts]
    imgs.sort()
    if len(imgs) == 0:
        return []
    return imgs if len(imgs) <= n else random.sample(imgs, n)

def compute_roi_bbox_for_viz(pil_img: Image.Image):
    pred = tooth_labelmap(pil_img)
    roi = pred > 0
    h, w = roi.shape[:2]
    if int(roi.sum()) < MIN_ROI_PIXELS:
        return None
    ys, xs = np.where(roi)
    if len(xs) == 0:
        return None
    x1, y1, x2, y2 = int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())
    return expand_asym(
        x1, y1, x2, y2, w=w, h=h,
        left=EXPAND_LEFT, right=EXPAND_RIGHT,
        top=EXPAND_TOP, bottom=EXPAND_BOTTOM
    )

def show_pairs_grid(title, src_dst_pairs, cols=2):
    if len(src_dst_pairs) == 0:
        print(title, "=> (no images)")
        return

    rows = len(src_dst_pairs)
    plt.figure(figsize=(cols * 5.2, rows * 4.2))
    plt.suptitle(title, fontsize=14)

    for i, (src_p, dst_p) in enumerate(src_dst_pairs):
        pil_src = Image.open(src_p).convert("RGB")
        src_np = np.array(pil_src).copy()
        bb = compute_roi_bbox_for_viz(pil_src)
        if bb is not None:
            x1, y1, x2, y2 = bb
            cv2.rectangle(src_np, (x1, y1), (x2, y2), (0, 255, 0), 3)

        pil_dst = Image.open(dst_p).convert("RGB")

        ax1 = plt.subplot(rows, cols, i * cols + 1)
        ax1.imshow(src_np)
        ax1.axis("off")
        ax1.set_title(f"ORIG (+bbox)\n{src_p.name}", fontsize=10)

        ax2 = plt.subplot(rows, cols, i * cols + 2)
        ax2.imshow(pil_dst)
        ax2.axis("off")
        ax2.set_title(f"ROI OUTPUT\n{dst_p.name}", fontsize=10)

    plt.tight_layout()
    plt.show()

for sp in splits:
    sp_root = Path(DST_ROOT) / sp
    if not sp_root.exists():
        continue

    classes = list_classes(sp_root)
    print(f"\n===== [{sp}] classes: {len(classes)} =====")

    for cls in classes:
        dst_class_dir = sp_root / cls
        picks_dst = sample_images(dst_class_dir, n=N_SHOW)

        pairs = []
        for dst_p in picks_dst:
            rel = dst_p.relative_to(Path(DST_ROOT))
            src_p = Path(SRC_ROOT) / rel
            if src_p.exists():
                pairs.append((src_p, dst_p))

        show_pairs_grid(f"{sp} / {cls} | ORIG vs ROI (n={len(pairs)})", pairs)

## segmentation ROI 데이터셋 학습 / 평가

ROI 버전 데이터셋에서 다시 3개 CNN 백본을 비교해  
재분할만 했을 때와 추가 ROI 전처리까지 했을 때의 차이를 확인합니다.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision.models import VGG19_BN_Weights, ResNet50_Weights, DenseNet121_Weights
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler

import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# 재분할 데이터셋만 사용
# ============================================================
from pathlib import Path

# 올바른 재분할 폴더(개수 100% 매칭)
#DATA_ROOT = "/content/drive/MyDrive/치아질환분류데이터셋_resplit_classwise_countMatched_v1"
DATA_ROOT = globals().get("DST_ROOT", "/content/drive/MyDrive/치아질환분류데이터셋_resplit_ROI_20260302-093923")
#  원본과 split 개수가 같은지 강제 체크
ORIG_ROOT = "/content/drive/MyDrive/치아질환분류데이터셋"

def count_imgs(d):
    exts = {".jpg",".jpeg",".png",".bmp",".webp"}
    d = Path(d)
    return sum(1 for p in d.rglob("*") if p.is_file() and p.suffix.lower() in exts)

orig_counts = {sp: count_imgs(f"{ORIG_ROOT}/{sp}") for sp in ["train","val","test"]}
res_counts  = {sp: count_imgs(f"{DATA_ROOT}/{sp}") for sp in ["train","val","test"]}

print(" ORIG counts:", orig_counts)
print(" RESPLIT(countMatched) counts:", res_counts)

assert orig_counts == res_counts, f"Split counts mismatch! ORIG={orig_counts}, RESPLIT={res_counts}"

# ============================================================
#  Drive 저장 폴더(재분할 전용)
# ============================================================
OUT_DIR = "/content/drive/MyDrive/model_compare_bestft_fixedclass/seg_roi"
os.makedirs(OUT_DIR, exist_ok=True)
print(" OUT_DIR =", OUT_DIR)

# ============================================================
#  클래스 순서 고정
# ============================================================
FIXED_CLASSES = ["calculus", "caries", "discoloration", "hypodontia", "ulcers"]

class RemapTargetsDataset(torch.utils.data.Dataset):
    def __init__(self, base_ds, fixed_classes):
        self.base = base_ds
        self.fixed_classes = list(fixed_classes)
        self.fixed_class_to_idx = {c:i for i,c in enumerate(self.fixed_classes)}

        ds_set = set(base_ds.classes)
        fixed_set = set(self.fixed_classes)
        if ds_set != fixed_set:
            missing = sorted(list(fixed_set - ds_set))
            extra = sorted(list(ds_set - fixed_set))
            raise ValueError(f"[class mismatch] missing={missing}, extra={extra}. base_ds.classes={base_ds.classes}")

        inv_old = {old_idx: cls for cls, old_idx in base_ds.class_to_idx.items()}
        self.old_to_fixed = {old_idx: self.fixed_class_to_idx[inv_old[old_idx]] for old_idx in inv_old}

        self.classes = self.fixed_classes
        self.class_to_idx = self.fixed_class_to_idx
        self.targets = [self.old_to_fixed[int(t)] for t in base_ds.targets]

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        x, y_old = self.base[idx]
        y = self.old_to_fixed[int(y_old)]
        return x, y

# ============================================================
# transforms
# ============================================================
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.0),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

eval_transforms = transforms.Compose([
    transforms.Resize(size=(256)),
    transforms.CenterCrop(size=(224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# ============================================================
# dataset 로드 (raw -> fixed mapping)
# ============================================================
train_ds_raw = datasets.ImageFolder(f"{DATA_ROOT}/train", transform=train_transforms)
val_ds_raw   = datasets.ImageFolder(f"{DATA_ROOT}/val",   transform=eval_transforms)
test_ds_raw  = datasets.ImageFolder(f"{DATA_ROOT}/test",  transform=eval_transforms)

train_ds = RemapTargetsDataset(train_ds_raw, FIXED_CLASSES)
val_ds   = RemapTargetsDataset(val_ds_raw, FIXED_CLASSES)
test_ds  = RemapTargetsDataset(test_ds_raw, FIXED_CLASSES)

class_names = train_ds.classes
num_classes = len(class_names)

print("DATA_ROOT:", DATA_ROOT)
print("FIXED class order:", class_names)
print("fixed class_to_idx:", train_ds.class_to_idx)

# ============================================================
# sampler 유지
# ============================================================
labels = torch.tensor(train_ds.targets, dtype=torch.long)
class_count = torch.bincount(labels, minlength=num_classes).float().clamp_min(1.0)
alpha = 1.0
class_weight = (1.0 / class_count) ** alpha
sample_weight = class_weight[labels]

SEED = 42
g = torch.Generator(); g.manual_seed(SEED)

train_sampler = WeightedRandomSampler(
    weights=sample_weight,
    num_samples=len(sample_weight),
    replacement=True,
    generator=g
)

train_loader = DataLoader(train_ds, batch_size=32, sampler=train_sampler, shuffle=False, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

# ============================================================
# 모델별 fine-tuning 정책 (VGG 기존 / ResNet,DenseNet best-ft)
# ============================================================
def freeze_all(net):
    for p in net.parameters():
        p.requires_grad = False

def enable_bn_affine(net):
    for m in net.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            if m.weight is not None: m.weight.requires_grad = True
            if m.bias is not None:   m.bias.requires_grad = True

def build_vgg19_bn(num_classes):
    net = models.vgg19_bn(weights=VGG19_BN_Weights.DEFAULT)
    freeze_all(net)
    enable_bn_affine(net)
    net.classifier[6] = nn.Linear(4096, num_classes)
    for p in net.classifier[6].parameters():
        p.requires_grad = True
    return net

def build_resnet50_bestft(num_classes):
    net = models.resnet50(weights=ResNet50_Weights.DEFAULT)
    freeze_all(net)
    for p in net.layer4.parameters():
        p.requires_grad = True
    enable_bn_affine(net)
    net.fc = nn.Linear(net.fc.in_features, num_classes)
    for p in net.fc.parameters():
        p.requires_grad = True
    return net

def build_densenet121_bestft(num_classes):
    net = models.densenet121(weights=DenseNet121_Weights.DEFAULT)
    freeze_all(net)
    for p in net.features.denseblock4.parameters():
        p.requires_grad = True
    if hasattr(net.features, "norm5"):
        for p in net.features.norm5.parameters():
            p.requires_grad = True
    enable_bn_affine(net)
    net.classifier = nn.Linear(net.classifier.in_features, num_classes)
    for p in net.classifier.parameters():
        p.requires_grad = True
    return net

def make_sgd_param_groups(net, head_keys, backbone_keys,
                          lr_head=1e-3, lr_backbone=1e-4, lr_bn=3e-4,
                          weight_decay=5e-4, momentum=0.9):
    head_params, backbone_params, bn_params = [], [], []
    for name, p in net.named_parameters():
        if not p.requires_grad:
            continue
        if "bn" in name.lower() or "batchnorm" in name.lower():
            bn_params.append(p)
            continue
        if any(k in name for k in head_keys):
            head_params.append(p)
        elif any(k in name for k in backbone_keys):
            backbone_params.append(p)
        else:
            backbone_params.append(p)

    param_groups = []
    if head_params:
        param_groups.append({"params": head_params, "lr": lr_head, "weight_decay": weight_decay, "momentum": momentum})
    if backbone_params:
        param_groups.append({"params": backbone_params, "lr": lr_backbone, "weight_decay": weight_decay, "momentum": momentum})
    if bn_params:
        param_groups.append({"params": bn_params, "lr": lr_bn, "weight_decay": 0.0, "momentum": momentum})

    return optim.SGD(param_groups, lr=lr_head, momentum=momentum, weight_decay=weight_decay)

# ============================================================
# TEST 평가
# ============================================================
@torch.no_grad()
def evaluate_on_loader(net, loader, criterion, num_classes, device):
    net.eval()
    cm = torch.zeros((num_classes, num_classes), dtype=torch.int64)
    total_loss, total_n = 0.0, 0

    for x, y in loader:
        x = x.to(device); y = y.to(device)
        logits = net(x)
        loss = criterion(logits, y)
        bs = y.size(0)
        total_loss += float(loss.item()) * bs
        total_n += bs

        pred = logits.argmax(dim=1)
        for t, p in zip(y.view(-1), pred.view(-1)):
            cm[t.long(), p.long()] += 1

    cm_f = cm.float()
    tp = torch.diag(cm_f)
    fn = cm_f.sum(dim=1) - tp
    fp = cm_f.sum(dim=0) - tp

    eps = 1e-12
    recall = tp / (tp + fn + eps)
    precision = tp / (tp + fp + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)

    macro_f1 = float(f1.mean().item())
    balanced_acc = float(recall.mean().item())
    acc = float(tp.sum().item() / max(cm_f.sum().item(), 1.0))
    avg_loss = total_loss / max(total_n, 1)

    return cm.cpu().numpy(), recall.cpu().numpy(), acc, macro_f1, balanced_acc, avg_loss

def normalize_rows(cm):
    row_sum = cm.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1
    return cm / row_sum

criterion = nn.CrossEntropyLoss()

# ============================================================
# 학습 + 리포트 생성 (resplit 전용)
# ============================================================
MODELS = [
    ("vgg19_bn", build_vgg19_bn),
    ("resnet50", build_resnet50_bestft),
    ("densenet121", build_densenet121_bestft),
]

results = []

for name, builder in MODELS:
    print("\n" + "="*80)
    print(f"Training: {name} (resplit)")
    print("="*80)

    net = builder(num_classes).to(device)

    if name == "vgg19_bn":
        optimizer = None
    elif name == "resnet50":
        optimizer = make_sgd_param_groups(net, head_keys=["fc."], backbone_keys=["layer4."],
                                          lr_head=1e-3, lr_backbone=1e-4, lr_bn=3e-4)
    else:
        optimizer = make_sgd_param_groups(net, head_keys=["classifier."],
                                          backbone_keys=["features.denseblock4.", "features.norm5."],
                                          lr_head=1e-3, lr_backbone=1e-4, lr_bn=3e-4)

    ckpt_path = os.path.join(OUT_DIR, f"best_{name}_seg_roi.pt")

    train_model_v2(
        optimizer_name="SGD",
        net=net,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        num_epochs=150,
        device=device,
        ckpt_path=ckpt_path,
        use_cutmix=False,
        use_mixup=False,
        optimizer=optimizer,
        default_lr=3e-5
    )

    ckpt = torch.load(ckpt_path, map_location=device)
    if "model_state_dict" in ckpt:
        net.load_state_dict(ckpt["model_state_dict"], strict=True)
    else:
        net.load_state_dict(ckpt["model"], strict=True)

    cm, recall, acc, macro_f1, bal_acc, loss = evaluate_on_loader(net, test_loader, criterion, num_classes, device)
    cm_norm = normalize_rows(cm)

    recall_str = " | ".join([f"{cls}:{recall[i]:.3f}" for i, cls in enumerate(class_names)])
    print(f"[{name}/resplit] TEST loss={loss:.4f} | acc={acc*100:.2f}% | macroF1={macro_f1:.4f} | bal_acc={bal_acc*100:.2f}%")
    print(f"[{name}/resplit] TEST Per-class Recall: {recall_str}")

    results.append({
        "model": name,
        "cm_norm": cm_norm,
        "recall": recall,
        "acc": acc,
        "macro_f1": macro_f1,
        "bal_acc": bal_acc,
        "loss": loss
    })

# 리포트 저장(덮어쓰기 방지: resplit 태그)
table_cols = ["Class"] + [r["model"] for r in results]
table_data = []
for i, cls in enumerate(class_names):
    table_data.append([cls] + [f"{r['recall'][i]:.3f}" for r in results])

overall_cols = ["Metric"] + [r["model"] for r in results]
overall_data = [
    ["Test Loss"] + [f"{r['loss']:.4f}" for r in results],
    ["Test Acc"] + [f"{r['acc']*100:.2f}%" for r in results],
    ["Test Macro-F1"] + [f"{r['macro_f1']:.4f}" for r in results],
    ["Test Balanced Acc"] + [f"{r['bal_acc']*100:.2f}%" for r in results],
]

fig = plt.figure(figsize=(18, 11))
for idx, r in enumerate(results):
    ax = fig.add_subplot(2, 3, idx+1)
    im = ax.imshow(r["cm_norm"], aspect="auto")
    ax.set_title(f"{r['model']} (resplit) normalized CM")
    ax.set_xlabel("Pred"); ax.set_ylabel("True")
    ax.set_xticks(range(num_classes)); ax.set_yticks(range(num_classes))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)

    for i in range(num_classes):
        for j in range(num_classes):
            ax.text(j, i, f"{r['cm_norm'][i,j]:.2f}", ha="center", va="center", fontsize=7)

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax_tbl = fig.add_subplot(2, 1, 2)
ax_tbl.axis("off")

tbl1 = ax_tbl.table(cellText=table_data, colLabels=table_cols, loc="upper center")
tbl1.auto_set_font_size(False); tbl1.set_fontsize(10); tbl1.scale(1.0, 1.25)

tbl2 = ax_tbl.table(cellText=overall_data, colLabels=overall_cols, loc="lower center")
tbl2.auto_set_font_size(False); tbl2.set_fontsize(11); tbl2.scale(1.0, 1.35)

ax_tbl.set_title("TEST Report (resplit): Per-class Recall + Overall Metrics", pad=10)

plt.tight_layout()
pdf_path = os.path.join(OUT_DIR, "report_onepage_seg_roi.pdf")
png_path = os.path.join(OUT_DIR, "report_onepage_seg_roi.png")
plt.savefig(pdf_path)
plt.savefig(png_path, dpi=220)
plt.close()

print("\n Saved report to Drive:")
print(" -", pdf_path)
print(" -", png_path)
print(" Saved ckpts to Drive inside:", OUT_DIR)

In [ ]:
# =========================
# Grad-CAM 시각화: 원본 + 모델별 Grad-CAM 비교 그리드
# =========================
from pathlib import Path
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image
import matplotlib.pyplot as plt

from torchvision import transforms
from torchvision.datasets import ImageFolder
import torchvision.models as models
import time

# =========================================
# 0) Config
# =========================================
DATA_ROOT = globals().get("DST_ROOT", "/content/drive/MyDrive/치아질환분류데이터셋_resplit_ROI_20260302-093923")
TEST_DIR = f"{DATA_ROOT}/test"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)

MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

IMG_SIZE = 224

#  SAVE_DIR에 타임스탬프 추가 및 dataset 태그 포함
SAVED_DATASET_TAG = Path(DATA_ROOT).name # 예: 치아질환분류데이터셋_resplit_ROI_20260302-093923
SAVED_DATE_TAG = time.strftime("%Y%m%d-%H%M%S")
SAVE_DIR = f"/content/drive/MyDrive/model_compare_bestft_fixedclass/seg_roi/gradcam_{SAVED_DATASET_TAG}_{SAVED_DATE_TAG}"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f" Grad-CAM save directory: {SAVE_DIR}")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================
# 1) Dataset
# =========================================
test_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

test_ds = ImageFolder(TEST_DIR, transform=test_tf)
class_to_idx = test_ds.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}

def find_discoloration_idx(class_to_idx: dict):
    for k, v in class_to_idx.items():
        if k.lower() == "discoloration":
            return v, k
    for k, v in class_to_idx.items():
        if "discolor" in k.lower():
            return v, k
    raise ValueError(f"Cannot find discoloration class in: {list(class_to_idx.keys())}")

DIS_IDX, DIS_NAME = find_discoloration_idx(class_to_idx)
print(" Discoloration class:", DIS_NAME, "->", DIS_IDX)

dis_samples = [(i, p, y) for i, (p, y) in enumerate(test_ds.samples) if y == DIS_IDX]
print(" Discoloration test samples:", len(dis_samples))
assert len(dis_samples) > 0, "test에 discoloration 샘플이 0개입니다."

# =========================================
# 2) Grad-CAM
# =========================================
def find_last_conv_layer(model: nn.Module):
    last_name, last_module = None, None
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            last_name, last_module = name, module
    if last_module is None:
        raise ValueError("No Conv2d layer found.")
    return last_name, last_module

class GradCAM:
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None

        def fwd_hook(module, inp, out):
            self.activations = out

        def bwd_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0]

        self.h1 = self.target_layer.register_forward_hook(fwd_hook)
        self.h2 = self.target_layer.register_full_backward_hook(bwd_hook)

    def remove_hooks(self):
        self.h1.remove()
        self.h2.remove()

    @torch.enable_grad() #이함수 내 에서는 반드시 그래디언트 계산 허용
    def generate(self, x: torch.Tensor, target_class: int):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        score = logits[:, target_class].sum()
        score.backward(retain_graph=True)

        A = self.activations
        dYdA = self.gradients

        weights = dYdA.mean(dim=(2,3), keepdim=True)
        cam = (weights * A).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(x.shape[2], x.shape[3]), mode="bilinear", align_corners=False)

        cam = cam.squeeze().detach()
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-12)
        return cam, logits.detach()

def unnormalize(img_t: torch.Tensor, mean=MEAN, std=STD):
    mean = torch.tensor(mean, device=img_t.device).view(3,1,1)
    std = torch.tensor(std, device=img_t.device).view(3,1,1)
    x = img_t * std + mean
    x = x.clamp(0,1)
    return x.permute(1,2,0).cpu().numpy()

def overlay_cam(rgb01: np.ndarray, cam01: np.ndarray, alpha=0.45):
    cmap = plt.get_cmap("jet")
    heat = cmap(cam01)[...,:3]
    out = (1 - alpha) * rgb01 + alpha * heat
    return np.clip(out, 0, 1)

# =========================================
# 3) Model build/load
# =========================================
def build_model(arch: str, num_classes: int):
    arch = arch.lower()
    if arch == "vgg19_bn":
        m = models.vgg19_bn(weights=None)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
        return m
    if arch == "resnet50":
        m = models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m
    if arch == "densenet121":
        m = models.densenet121(weights=None)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
        return m
    raise ValueError(f"Unknown arch: {arch}")

def load_checkpoint(model: nn.Module, ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        sd = ckpt["model_state_dict"]
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        sd = ckpt["state_dict"]
    elif isinstance(ckpt, dict):
        sd = ckpt
    else:
        raise ValueError("Unsupported checkpoint format.")

    new_sd = {k.replace("module.",""): v for k, v in sd.items()}
    model.load_state_dict(new_sd, strict=True)
    return model

MODEL_SPECS = [
    {"name": "VGG19_BN", "arch": "vgg19_bn", "ckpt": "/content/drive/MyDrive/model_compare_bestft_fixedclass/seg_roi/best_vgg19_bn_seg_roi.pt"},
    {"name": "ResNet50", "arch": "resnet50", "ckpt": "/content/drive/MyDrive/model_compare_bestft_fixedclass/seg_roi/best_resnet50_seg_roi.pt"},
    {"name": "DenseNet121", "arch": "densenet121", "ckpt": "/content/drive/MyDrive/model_compare_bestft_fixedclass/seg_roi/best_densenet121_seg_roi.pt"},
]

NUM_CLASSES = len(class_to_idx)

# 모델들 미리 로드 + 각 모델별 Grad-CAM 준비
models_dict = {}
for spec in MODEL_SPECS:
    m = build_model(spec["arch"], NUM_CLASSES)
    m = load_checkpoint(m, spec["ckpt"])
    m = m.to(device).eval()

    layer_name, layer_module = find_last_conv_layer(m)
    print(f" {spec['name']} target layer:", layer_name)
    cam_engine = GradCAM(m, layer_module)

    models_dict[spec["name"]] = {"model": m, "cam": cam_engine}

# =========================================
# 4) 한 화면 비교 시각화 (rows=images, cols=원본+모델들)
# =========================================
alpha = 0.45
target_class = DIS_IDX

n_imgs = len(dis_samples)          # 18 예상
n_models = len(MODEL_SPECS)
n_cols = n_models + 1              #  원본 컬럼(1개) 추가

#  figure 가로폭도 컬럼 수에 맞춰 확장
fig = plt.figure(figsize=(n_cols * 4.6, n_imgs * 2.6))

for r, (idx, img_path, y) in enumerate(dis_samples):
    x, _ = test_ds[idx]
    x1 = x.unsqueeze(0).to(device)
    rgb01 = unnormalize(x.to(device))

    base = os.path.splitext(os.path.basename(img_path))[0]

    # -------------------------
    #  원본 이미지 컬럼
    # -------------------------
    ax0 = plt.subplot(n_imgs, n_cols, r * n_cols + 1)   # col=0
    ax0.imshow(rgb01)
    ax0.axis("off")
    if r == 0:
        ax0.set_title("Original", fontsize=10)          #  첫 줄 헤더

    # 이미지 id 표시는 원본 칸에 표시(가장 직관적)
    ax0.text(3, 12, base, color="white", fontsize=8,
             bbox=dict(facecolor="black", alpha=0.55, pad=2))

    # -------------------------
    # (B) 모델별 Grad-CAM 컬럼들
    # -------------------------
    for m_i, spec in enumerate(MODEL_SPECS):
        name = spec["name"]
        m = models_dict[name]["model"]
        cam_engine = models_dict[name]["cam"]

        with torch.no_grad():
            logits0 = m(x1)
            prob = F.softmax(logits0, dim=1)[0]
            pred = int(torch.argmax(prob).item())
            pred_name = idx_to_class[pred]
            pred_p = float(prob[pred].item())
            dis_p = float(prob[DIS_IDX].item())
            correct = (pred == DIS_IDX)

        cam01, _ = cam_engine.generate(x1, target_class=target_class)
        over = overlay_cam(rgb01, cam01.cpu().numpy(), alpha=alpha)

        #  col index: 원본(0) 다음이 첫 모델(1)
        col = 1 + m_i
        ax = plt.subplot(n_imgs, n_cols, r * n_cols + col + 1)
        ax.imshow(over)
        ax.axis("off")

        # 첫 줄(row=0)에만 모델 이름 헤더처럼
        if r == 0:
            ax.set_title(f"{name}\nPred={pred_name}({pred_p:.2f}) P(dis)={dis_p:.2f}", fontsize=9)
        else:
            ax.set_title(f"Pred={pred_name}({pred_p:.2f}) P(dis)={dis_p:.2f} | ok={correct}", fontsize=8)

grid_path = os.path.join(SAVE_DIR, "GRID__ORIG_plus_compare_VGG_ResNet_DenseNet__target_discoloration.png")  ## 파일명
plt.tight_layout()
plt.savefig(grid_path, dpi=200)
plt.show()

# hook 해제
for name in models_dict:
    models_dict[name]["cam"].remove_hooks()

print(" saved:", grid_path)

In [ ]:
from IPython.display import Image as IPyImage, display, IFrame
from pathlib import Path

SEG_OUT_DIR = Path("/content/drive/MyDrive/model_compare_bestft_fixedclass/seg_roi")
png_path = SEG_OUT_DIR / "report_onepage_seg_roi.png"
pdf_path = SEG_OUT_DIR / "report_onepage_seg_roi.pdf"

print("PNG exists?", png_path.exists(), png_path)
print("PDF exists?", pdf_path.exists(), pdf_path)

if png_path.exists():
    display(IPyImage(filename=str(png_path)))
if pdf_path.exists():
    display(IFrame(src=str(pdf_path), width=1100, height=750))

# 부록. 아카이브 실험 / 제거한 중복 블록

 
아래는 과정 중에 사용했던 실험 흔적이나, 현재 메인 노트북에서는 제외한 블록들입니다.

본문에서 제외한 대표 항목은 다음과 같습니다.

- legacy 개별 모델 학습 / test / confusion matrix 블록  
  → 최신 통합 비교 셀로 대체
- 외부 단일 폴더 검증(FlatImageDataset)  
  → 경로 / target class를 다시 점검해야 해서 메인에서 제외
- `file_kb`, `brightness_mean` 기준 대체 재분할  
  → sharpness 기반 핵심 흐름 외의 민감도 분석이라 부록으로 처리
- YOLO ROI 실험  
  → segmentation ROI 대비 탐색적 성격이 강해 부록으로 분리

## 이전 watch-on / early stopping 경보 기준 실험
아래 코드는 당시 실험 흔적을 그대로 보존한 아카이브 블록입니다.

In [ ]:
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from tqdm import tqdm

# # =========================
# # confusion matrix 기반 지표 계산 함수 (기존 유지)
# # =========================
# def metrics_from_confusion_matrix(cm: torch.Tensor):
#     cm_f = cm.to(dtype=torch.float32)

#     tp = torch.diag(cm_f)
#     fn = cm_f.sum(dim=1) - tp
#     fp = cm_f.sum(dim=0) - tp

#     eps = 1e-12
#     recall = tp / (tp + fn + eps)
#     precision = tp / (tp + fp + eps)

#     balanced_acc = recall.mean().item()

#     f1 = 2 * precision * recall / (precision + recall + eps)
#     macro_f1 = f1.mean().item()

#     per_class_recall = recall.detach().cpu().tolist()
#     return macro_f1, balanced_acc, per_class_recall


# # =========================
# # ✅ 변경: "best 갱신 + no-improve 카운터" 전용 트래커
# # =========================
# class BestTracker:
#     def __init__(self, mode: str, min_delta: float, name: str):
#         assert mode in ["min", "max"]
#         self.mode = mode
#         self.min_delta = float(min_delta)
#         self.name = name
#         self.best = None
#         self.best_epoch = None

#     def improved(self, x: float) -> bool:
#         if self.best is None:
#             return True
#         if self.mode == "min":
#             return x < (self.best - self.min_delta)
#         else:
#             return x > (self.best + self.min_delta)

#     def update(self, x: float, epoch: int):
#         """return (is_improved, prev_best)"""
#         x = float(x)
#         prev = self.best
#         if self.improved(x):
#             self.best = x
#             self.best_epoch = epoch
#             return True, prev
#         return False, prev


# # =========================
# # ✅ 변경: best ckpt 저장(주 지표=macro-F1 기준)
# # =========================
# def save_ckpt(path, epoch, model, optimizer, scheduler, best_val_loss, best_macro_f1,
#               best_bal_acc, best_per_class_recall, class_names):
#     torch.save({
#         "epoch": epoch,
#         "model": model.state_dict(),
#         "optimizer": optimizer.state_dict() if optimizer else None,
#         "scheduler": scheduler.state_dict() if scheduler else None,
#         "best_val_loss": best_val_loss,
#         "best_macro_f1": best_macro_f1,
#         "best_balanced_acc": best_bal_acc,
#         "best_per_class_recall": best_per_class_recall,
#         "class_names": class_names,
#     }, path)


# def load_ckpt(path, model, optimizer=None, scheduler=None, device="cpu"):
#     ckpt = torch.load(path, map_location=device)
#     model.load_state_dict(ckpt["model"])
#     if optimizer is not None and ckpt.get("optimizer") is not None:
#         optimizer.load_state_dict(ckpt["optimizer"])
#     if scheduler is not None and ckpt.get("scheduler") is not None:
#         scheduler.load_state_dict(ckpt["scheduler"])
#     return ckpt


# # =========================
# # ✅ 변경: "WATCH(val_loss 정체) -> OVERFIT(macro-F1 정체/악화) -> 즉시 종료" v3
# # =========================
# def train_model_v2(
#     optimizer_name,
#     net,
#     train_loader,
#     val_loader,
#     criterion,
#     num_epochs,
#     scheduler=None,
#     ckpt_path="best_v3.pt",
#     start_epoch=0,
#     device=None,
#     print_per_class_recall=True,

#     # ✅ 기본값(네 요구 반영)
#     warmup_epochs=50,          # (default=50) warmup 동안은 경보/중지 금지
#     watch_patience=6,          # (default=6) val_loss best 갱신 없음 N
#     overfit_patience=6,        # (default=6) WATCH 중 macro-F1 best 갱신 없음 M

#     # ✅ "정체" 판단을 위해 min_delta를 둠 (너의 핵심 요구)
#     loss_min_delta=0.002,      # (default=0.002) val_loss best 갱신 최소 개선폭
#     f1_min_delta=0.002,        # (default=0.002) macro-F1 best 갱신 최소 개선폭

#     # ✅ "또는 악화" 옵션 (너가 원함) — 너무 민감하면 끄거나 delta를 키워라
#     f1_worsen_delta=0.005,     # (default=0.005) best 대비 이만큼 떨어지면 악화로 간주
#     f1_worsen_patience=2,      # (default=2) 연속 악화 횟수

#     stop_on_overfit=True,      # (default=True) OVERFIT 뜨면 즉시 종료(=EarlyStop)
# ):
#     # device 자동
#     if device is None:
#         device = next(net.parameters()).device if any(True for _ in net.parameters()) \
#                  else torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     net.to(device)

#     # optimizer
#     if optimizer_name == "SGD":
#         optimizer = optim.SGD(net.parameters(), lr=3e-5, momentum=0.9, weight_decay=5e-4)
#     elif optimizer_name == "Adam":
#         optimizer = optim.Adam(net.parameters(), lr=3e-5, betas=(0.9, 0.999))
#     elif optimizer_name == "RAdam":
#         optimizer = optim.RAdam(net.parameters(), lr=3e-5, betas=(0.9, 0.999))
#     else:
#         raise ValueError(f"Unsupported optimizer: {optimizer_name}")

#     class_names = train_loader.dataset.classes
#     num_classes = len(class_names)

#     # logs
#     train_losses, val_losses = [], []
#     val_accuracies, val_macro_f1s, val_balanced_accuracies = [], [], []
#     val_per_class_recalls = []
#     gap_history = []  # ✅ 변경: gap은 시각화/진단용으로만 저장

#     # ✅ 변경: best 트래커(갱신 로그 출력용)
#     val_loss_tracker = BestTracker(mode="min", min_delta=loss_min_delta, name="val_loss")
#     f1_tracker = BestTracker(mode="max", min_delta=f1_min_delta, name="macro_f1")

#     # WATCH/OVERFIT 상태
#     watch_on = False
#     no_improve_val_loss = 0

#     # WATCH 구간에서만 F1을 따로 카운트(“WATCH 상태에서 M epoch” 요구 반영)
#     watch_no_improve_f1 = 0
#     watch_worse_f1_streak = 0

#     best_val_loss_for_ckpt = None
#     best_f1_for_ckpt = None
#     best_bal_acc_for_ckpt = None
#     best_recall_for_ckpt = None

#     for epoch in range(start_epoch, num_epochs):
#         # -------- Train --------
#         net.train()
#         running_loss = 0.0
#         for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [train]"):
#             inputs, labels = inputs.to(device), labels.to(device)
#             optimizer.zero_grad()
#             outputs = net(inputs)
#             loss = criterion(outputs, labels)
#             loss.backward()
#             optimizer.step()
#             running_loss += loss.item()

#         train_loss = running_loss / max(1, len(train_loader))
#         train_losses.append(train_loss)

#         # -------- Val --------
#         net.eval()
#         val_loss_sum = 0.0
#         correct, total = 0, 0
#         cm = torch.zeros((num_classes, num_classes), device=device, dtype=torch.int64)

#         with torch.no_grad():
#             for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [val]"):
#                 inputs, labels = inputs.to(device), labels.to(device)
#                 outputs = net(inputs)
#                 loss = criterion(outputs, labels)
#                 val_loss_sum += loss.item()

#                 predicted = outputs.argmax(dim=1)
#                 total += labels.size(0)
#                 correct += (predicted == labels).sum().item()

#                 idx = labels * num_classes + predicted
#                 cm += torch.bincount(idx, minlength=num_classes * num_classes).reshape(num_classes, num_classes)

#         val_loss = val_loss_sum / max(1, len(val_loader))
#         val_losses.append(val_loss)

#         val_acc = 100.0 * correct / max(1, total)
#         val_accuracies.append(val_acc)

#         val_macro_f1, val_bal_acc, per_class_recall = metrics_from_confusion_matrix(cm)
#         val_macro_f1s.append(val_macro_f1)
#         val_balanced_accuracies.append(val_bal_acc)
#         val_per_class_recalls.append(per_class_recall)

#         # ✅ 변경: gap은 기록만
#         gap_history.append(max(0.0, float(val_loss - train_loss)))

#         print(
#             f'[{optimizer_name}] Epoch {epoch+1}, '
#             f'Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}, '
#             f'Val Acc: {val_acc:.2f}%, Val Macro-F1: {val_macro_f1:.4f}, '
#             f'Val Balanced Acc: {val_bal_acc:.4f}'
#         )

#         if print_per_class_recall:
#             recall_str = " | ".join([f"{n}:{r:.3f}" for n, r in zip(class_names, per_class_recall)])
#             print(f"Per-class Recall: {recall_str}")

#         # =========================================================
#         # ✅ 변경: best 갱신 로그 출력 (val_loss / macro-F1 각각)
#         # =========================================================
#         # warmup 동안에도 best는 업데이트해도 됨(단, 경보/중지 로직은 warmup 이후)
#         val_loss_improved, prev_best_loss = val_loss_tracker.update(val_loss, epoch)
#         if val_loss_improved:
#             print(f"✅ BEST val_loss 업데이트: {val_loss_tracker.best:.6f} (epoch {epoch+1})")
#             no_improve_val_loss = 0  # ✅ 변경: best 갱신되면 카운터 리셋
#         else:
#             if (epoch + 1) > warmup_epochs:
#                 no_improve_val_loss += 1

#         f1_improved, prev_best_f1 = f1_tracker.update(val_macro_f1, epoch)
#         if f1_improved:
#             print(f"✅ BEST macro-F1 업데이트: {f1_tracker.best:.4f} (epoch {epoch+1})")

#             # ✅ 변경: 주 지표(=macro-F1) best일 때 ckpt 저장
#             best_val_loss_for_ckpt = val_loss_tracker.best
#             best_f1_for_ckpt = f1_tracker.best
#             best_bal_acc_for_ckpt = val_bal_acc
#             best_recall_for_ckpt = per_class_recall

#             save_ckpt(
#                 ckpt_path, epoch, net, optimizer, scheduler,
#                 best_val_loss_for_ckpt, best_f1_for_ckpt,
#                 best_bal_acc_for_ckpt, best_recall_for_ckpt, class_names
#             )

#         # =========================================================
#         # ✅ 변경: warmup 전에는 WATCH/OVERFIT 금지
#         # =========================================================
#         if (epoch + 1) <= warmup_epochs:
#             if scheduler is not None:
#                 # ReduceLROnPlateau면 보통 val_loss 넣어줌(선택)
#                 try:
#                     scheduler.step(val_loss)
#                 except TypeError:
#                     scheduler.step()
#             continue

#         # =========================================================
#         # ✅ 변경: WATCH(1차) — val_loss best 갱신이 N epoch 동안 없음
#         # =========================================================
#         if (not watch_on) and (no_improve_val_loss >= watch_patience):
#             watch_on = True
#             watch_no_improve_f1 = 0
#             watch_worse_f1_streak = 0
#             print(f"🟡 WATCH ON: val_loss best 갱신 없음 {watch_patience} epochs (warmup={warmup_epochs} 이후)")

#         # WATCH 해제: val_loss가 다시 best 갱신되면 OFF
#         if watch_on and val_loss_improved:
#             watch_on = False
#             watch_no_improve_f1 = 0
#             watch_worse_f1_streak = 0
#             print("🟢 WATCH OFF: val_loss가 다시 best 갱신되어 경보 해제")

#         # =========================================================
#         # ✅ 변경: OVERFIT(2차) — WATCH 상태에서 macro-F1 best 갱신 없음/악화
#         # =========================================================
#         overfit_trigger = False
#         if watch_on:
#             # 1) WATCH 중 macro-F1 best 갱신이 M epoch 동안 없음
#             if not f1_improved:
#                 watch_no_improve_f1 += 1
#             else:
#                 watch_no_improve_f1 = 0  # WATCH 중이라도 best 갱신되면 리셋

#             # 2) 또는 "악화" 연속 감지(옵션)
#             if (f1_tracker.best is not None) and (val_macro_f1 < (f1_tracker.best - f1_worsen_delta)):
#                 watch_worse_f1_streak += 1
#             else:
#                 watch_worse_f1_streak = 0

#             if (watch_no_improve_f1 >= overfit_patience) or (watch_worse_f1_streak >= f1_worsen_patience):
#                 overfit_trigger = True

#         if overfit_trigger:
#             print(
#                 f"🔴 OVERFIT ON: WATCH 중 macro-F1 best 갱신 정체/악화 → 종료 (epoch {epoch+1}) "
#                 f"(no_improve_f1_in_watch={watch_no_improve_f1}/{overfit_patience}, "
#                 f"worse_streak={watch_worse_f1_streak}/{f1_worsen_patience})"
#             )
#             if stop_on_overfit:
#                 break

#         # scheduler step
#         if scheduler is not None:
#             try:
#                 scheduler.step(val_loss)  # ReduceLROnPlateau 호환
#             except TypeError:
#                 scheduler.step()

#     # =========================
#     # ✅ 변경: best ckpt 로드(항상 best macro-F1으로 복원)
#     # =========================
#     ckpt = load_ckpt(ckpt_path, net, optimizer=optimizer, scheduler=scheduler, device=device)
#     best_epoch = ckpt["epoch"] + 1

#     return (
#         train_losses, val_losses, val_accuracies,
#         val_macro_f1s, val_balanced_accuracies, val_per_class_recalls,
#         gap_history,
#         best_epoch,
#         ckpt["best_macro_f1"],
#         ckpt["best_balanced_acc"],
#         ckpt["class_names"],
#         ckpt["best_per_class_recall"],
#         ckpt["best_val_loss"],
#     )

## ImageNet mean/std + BN 재학습 비교 실험
아래 코드는 당시 실험 흔적을 그대로 보존한 아카이브 블록입니다.

In [ ]:
# import torch
# import torch.nn as nn
# import torchvision.models as models
# from torchvision.models import VGG19_BN_Weights  # ✅ [유지] pretrained 경고 제거 + 최신 weights 사용
# from torchvision import datasets, transforms
# from torch.utils.data import DataLoader
# import torch.optim.lr_scheduler as lr_scheduler # 스케줄러 import 추가

# # Redefine the device (already defined but for completeness)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # =========================
# # ✅ [변경 1] ImageNet mean/std로 정규화 (pretrained가 기대하는 입력 분포로 맞춤)
# # =========================
# # mean= [0.714, 0.472, 0.419]
# # std = [0.274, 0.298, 0.282]
# mean = [0.485, 0.456, 0.406]
# std  = [0.229, 0.224, 0.225]

# #데이터 클래스에 영향을 주지않는 기본적인 데이터증강 적용 ->이를통해 먼저 recall=0인 문제점이 해결되는지 확인
# train_transforms = transforms.Compose([
#     transforms.RandomResizedCrop(224, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
#     transforms.RandomHorizontalFlip(p=0.5),
#     transforms.RandomRotation(degrees=10),
#     # 색은 "아주 약하게"만 (선택)
#     transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.0),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=mean, std=std),
# ])

# val_transforms = transforms.Compose([
#     transforms.Resize(size=(256)),
#     transforms.CenterCrop(size=(224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=mean, std=std)  # ✅ [변경]
# ])

# # Redefine dataset paths (gdrive_dataset_path is available from context)
# gdrive_dataset_path = '/content/drive/MyDrive/치아질환분류데이터셋'

# # Redefine datasets and data loaders
# train_dataset_v2 = datasets.ImageFolder(f"{gdrive_dataset_path}/train", transform=train_transforms)
# val_dataset_v2   = datasets.ImageFolder(f"{gdrive_dataset_path}/val",   transform=val_transforms)
# train_loader_v2 = DataLoader(train_dataset_v2, batch_size=32, shuffle=True)
# val_loader_v2   = DataLoader(val_dataset_v2, batch_size=32, shuffle=False)

# # =========================
# # ✅ [변경 2] pretrained=True 대신 weights 사용 (경고 제거)
# # =========================
# vggnet_v2 = models.vgg19_bn(weights=VGG19_BN_Weights.DEFAULT)  # ✅ [변경]

# # =========================
# # ✅ [변경 3] "BN은 학습"시키고, 나머지 백본은 freeze 유지
# #   - 네 주장(내 데이터셋에 맞게 BN 적응) 검증용
# #   - BN 파라미터(weight/bias)는 학습되게 하고
# #   - BN running stats는 train 모드에서 업데이트되도록 둠
# # =========================
# # (1) 우선 전부 freeze
# for param in vggnet_v2.parameters():
#     param.requires_grad = False

# # (2) ✅ [변경] BN 파라미터만 unfreeze
# for m in vggnet_v2.modules():
#     if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
#         if m.weight is not None:
#             m.weight.requires_grad = True  # ✅ [변경]
#         if m.bias is not None:
#             m.bias.requires_grad = True    # ✅ [변경]
#         # 주의: m.train()/m.eval()은 train_model_v2에서 net.train()로 결정됨
#         #      (=여기선 따로 건드리지 않음)

# # Replace the last layer of the classifier for 5 classes and unfreeze its parameters
# vggnet_v2.classifier[6] = nn.Linear(4096, 5)
# for param in vggnet_v2.classifier[6].parameters():
#     param.requires_grad = True

# vggnet_v2 = vggnet_v2.to(device)

# # Define the criterion(기준loss정의)
# criterion_v2 = nn.CrossEntropyLoss()

# # # Define a scheduler (commented out for optional use) 학습률 스케줄러 정의
# # scheduler_v2 = lr_scheduler.ReduceLROnPlateau(
# #     optimizer=torch.optim.SGD(vggnet_v2.parameters(), lr=3e-5, momentum=0.9, weight_decay=5e-4), # optimizer를 여기에 다시 정의해야 합니다.
# #     mode='min', factor=0.1, patience=5, verbose=True
# # )


# # Train the vggnet_v2 model using the new train_model_v2 function
# # UPDATED: Capture all 13 return values
# train_losses_v2, val_losses_v2, val_accuracies_v2, val_macro_f1s_v2, val_balanced_accuracies_v2, val_per_class_recalls_v2, ema_gap_history_v2, start_epoch_v2, best_val_macro_f1_v2, best_val_balanced_accuracy_v2, best_class_names_v2, best_per_class_recall_v2, best_val_loss_v2 = \
# train_model_v2('SGD', vggnet_v2, train_loader_v2, val_loader_v2, criterion_v2,
#                num_epochs=100, device=device, ckpt_path="best_full_v2.pt",
# #               scheduler=scheduler_v2) # 스케줄러 인자 주석 처리
# )

# print(f"\n최고 성능 (Early Stopping 기준 - Macro-F1):")
# print(f"  Best Validation Macro-F1: {best_val_macro_f1_v2:.4f}")
# print(f"  Best Validation Balanced Accuracy: {best_val_balanced_accuracy_v2*100:.2f}%")

# # “마지막 최고 성능 출력”에 per-class recall 추가
# if best_per_class_recall_v2 is not None:
#     best_recall_str = " | ".join([f"{n}:{r:.3f}" for n, r in zip(best_class_names_v2, best_per_class_recall_v2)])
#     print(f"  Best Per-class Recall: {best_recall_str}")
# else:
#     print("  Best Per-class Recall: (없음) - ckpt 저장값 확인 필요")

## WeightedRandomSampler 추가 실험
아래 코드는 당시 실험 흔적을 그대로 보존한 아카이브 블록입니다.

In [ ]:
# import torch
# import torch.nn as nn
# import torchvision.models as models
# from torchvision.models import VGG19_BN_Weights  # ✅ [유지] pretrained 경고 제거 + 최신 weights 사용
# from torchvision import datasets, transforms

# # ✅ [변경] WeightedRandomSampler를 쓰려면 여기서 같이 import 해야 함
# from torch.utils.data import DataLoader, WeightedRandomSampler

# import torch.optim.lr_scheduler as lr_scheduler  # 스케줄러 import 추가

# # Redefine the device (already defined but for completeness)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # =========================
# # ✅ [변경 1] ImageNet mean/std로 정규화 (pretrained가 기대하는 입력 분포로 맞춤)
# # =========================
# # mean= [0.714, 0.472, 0.419]
# # std = [0.274, 0.298, 0.282]
# mean = [0.485, 0.456, 0.406]
# std  = [0.229, 0.224, 0.225]

# # 데이터 클래스에 영향을 주지않는 기본적인 데이터증강 적용
# # -> 이를 통해 먼저 recall=0인 문제점이 해결되는지 확인
# train_transforms = transforms.Compose([
#     transforms.RandomResizedCrop(224, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
#     transforms.RandomHorizontalFlip(p=0.5),
#     transforms.RandomRotation(degrees=10),
#     # 색은 "아주 약하게"만 (선택)
#     transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.0),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=mean, std=std),
# ])

# val_transforms = transforms.Compose([
#     transforms.Resize(size=(256)),
#     transforms.CenterCrop(size=(224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=mean, std=std)  # ✅ [유지]
# ])

# # Redefine dataset paths (gdrive_dataset_path is available from context)
# gdrive_dataset_path = '/content/drive/MyDrive/치아질환분류데이터셋'

# # =========================
# # ✅ [유지] ImageFolder로 데이터셋 로드
# # =========================
# train_dataset_v2 = datasets.ImageFolder(f"{gdrive_dataset_path}/train", transform=train_transforms)
# val_dataset_v2   = datasets.ImageFolder(f"{gdrive_dataset_path}/val",   transform=val_transforms)

# # =========================
# # ✅ [추가 4] WeightedRandomSampler로 "학습 배치 분포"를 재균형
# # =========================
# # [핵심 배경]
# # - 지금 문제: discoloration이 test에서 calculus로 많이 빨려 들어감(= 다수 클래스로 결정이 기움)
# # - recall=0은 보통 "모델이 그 클래스를 전혀 못 봤다"기보다,
# #   학습 중 미니배치에 너무 적게 등장해서 SGD gradient 업데이트가 사실상 발생하지 않는 경우가 많음.
# # - WeightedRandomSampler는 "샘플 뽑는 확률"을 조절해서,
# #   소수 클래스(discoloration)가 학습 배치에 더 자주 들어오도록 만듦
# #   → 그 클래스에 대한 gradient가 누적 → decision boundary가 움직일 여지가 생김(딥러닝/SGD 관점).
# #
# # [중요 오해 방지]
# # - weights는 '클래스 가중치(클래스 개수 길이)'가 아니라
# #   '샘플별 가중치(len(train_dataset) 길이)'여야 함.
# #
# # [가중치 설계]
# # - 가장 강한 기본형: class_weight = 1 / class_count  (소수 클래스가 많이 뽑힘 → recall=0 깨기 우선)
# # - 과적합/중복이 걱정되면 완만하게: class_weight = 1 / sqrt(class_count)

# # ✅ [추가] train 라벨 벡터 (ImageFolder는 targets를 제공)
# labels = torch.tensor(train_dataset_v2.targets, dtype=torch.long)

# # ✅ [추가] 클래스 수(=5개) 안전하게 계산
# num_classes = len(train_dataset_v2.classes)

# # ✅ [추가] 클래스별 샘플 수 카운트
# class_count = torch.bincount(labels, minlength=num_classes).float()
# class_count = class_count.clamp_min(1.0)  # 0 나눗셈 방지(혹시 비어있는 클래스가 있을 때)

# # ✅ [추가] 클래스 가중치(역빈도) → 소수 클래스일수록 weight 큼
# # alpha를 쓰면 조절 가능: alpha=1.0(강), alpha=0.5(완만)
# alpha = 1.0  # ✅ [추가] 우선 recall=0을 깨는 게 목표면 1.0부터 시작 추천
# class_weight = (1.0 / class_count) ** alpha

# # ✅ [추가] "샘플별 가중치"로 확장 (길이 = len(train_dataset_v2))
# sample_weight = class_weight[labels]

# # ✅ [추가] 재현성을 원하면 generator에 시드 고정 가능 (generator 기본값=None)
# SEED = 42
# g = torch.Generator()
# g.manual_seed(SEED)

# # ✅ [추가] WeightedRandomSampler 생성
# # - num_samples: 한 epoch에서 뽑을 샘플 수 (보통 len(train_dataset)로 둬서 epoch 길이 유지)
# # - replacement: 기본값 True (중복추출 허용 = 오버샘플링 효과)
# # - generator: 기본값 None (여기선 재현성 위해 넣음)
# train_sampler = WeightedRandomSampler(
#     weights=sample_weight,
#     num_samples=len(sample_weight),
#     replacement=True,
#     generator=g
# )

# # ✅ [추가] 디버깅/레포트용: 클래스 분포와 가중치 출력(선택)
# print("Class names:", train_dataset_v2.classes)
# print("Train class_count:", class_count.tolist())
# print("Train class_weight(alpha={}):".format(alpha), class_weight.tolist())

# # =========================
# # ✅ [변경] train_loader에 sampler 적용
# # =========================
# # - sampler를 쓰면 "샘플 순서/선택"은 sampler가 책임지므로 shuffle=True를 같이 쓰면 안 됨
# # - 따라서 shuffle=False로 둬야 함
# train_loader_v2 = DataLoader(
#     train_dataset_v2,
#     batch_size=32,
#     sampler=train_sampler,   # ✅ [변경] shuffle 대신 sampler 사용
#     shuffle=False            # ✅ [변경] sampler 사용 시 shuffle=False
# )

# # ✅ [유지] val은 절대 sampler 적용하지 말 것(평가 분포를 바꾸면 공정성 붕괴)
# val_loader_v2 = DataLoader(val_dataset_v2, batch_size=32, shuffle=False)

# # =========================
# # ✅ [변경 2] pretrained=True 대신 weights 사용 (경고 제거)
# # =========================
# vggnet_v2 = models.vgg19_bn(weights=VGG19_BN_Weights.DEFAULT)  # ✅ [유지]

# # =========================
# # ✅ [변경 3] "BN은 학습"시키고, 나머지 백본은 freeze 유지
# # =========================
# # (1) 우선 전부 freeze
# for param in vggnet_v2.parameters():
#     param.requires_grad = False

# # (2) ✅ [유지] BN 파라미터만 unfreeze
# for m in vggnet_v2.modules():
#     if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
#         if m.weight is not None:
#             m.weight.requires_grad = True
#         if m.bias is not None:
#             m.bias.requires_grad = True
#         # running stats는 net.train()에서 업데이트됨

# # Replace the last layer of the classifier for 5 classes and unfreeze its parameters
# vggnet_v2.classifier[6] = nn.Linear(4096, 5)
# for param in vggnet_v2.classifier[6].parameters():
#     param.requires_grad = True

# vggnet_v2 = vggnet_v2.to(device)

# # Define the criterion(기준loss정의)
# criterion_v2 = nn.CrossEntropyLoss()

# # Train the vggnet_v2 model using the new train_model_v2 function
# train_losses_v2, val_losses_v2, val_accuracies_v2, val_macro_f1s_v2, val_balanced_accuracies_v2, val_per_class_recalls_v2, ema_gap_history_v2, start_epoch_v2, best_val_macro_f1_v2, best_val_balanced_accuracy_v2, best_class_names_v2, best_per_class_recall_v2, best_val_loss_v2 = \
# train_model_v2('SGD', vggnet_v2, train_loader_v2, val_loader_v2, criterion_v2,
#                num_epochs=100, device=device, ckpt_path="best_full_v2.pt",
# #               scheduler=scheduler_v2
# )

# print(f"\n최고 성능 (Early Stopping 기준 - Macro-F1):")
# print(f"  Best Validation Macro-F1: {best_val_macro_f1_v2:.4f}")
# print(f"  Best Validation Balanced Accuracy: {best_val_balanced_accuracy_v2*100:.2f}%")

# # “마지막 최고 성능 출력”에 per-class recall 추가
# if best_per_class_recall_v2 is not None:
#     best_recall_str = " | ".join([f"{n}:{r:.3f}" for n, r in zip(best_class_names_v2, best_per_class_recall_v2)])
#     print(f"  Best Per-class Recall: {best_recall_str}")
# else:
#     print("  Best Per-class Recall: (없음) - ckpt 저장값 확인 필요")

## CutMix 추가 실험
아래 코드는 당시 실험 흔적을 그대로 보존한 아카이브 블록입니다.

In [ ]:
# import torch
# import torch.nn as nn
# import torchvision.models as models
# from torchvision.models import VGG19_BN_Weights
# from torchvision import datasets, transforms
# from torch.utils.data import DataLoader, WeightedRandomSampler

# # ✅ [중요] 1번 파일에서 train_model_v2를 import 해서 사용한다고 가정
# # from model_train_v2 import train_model_v2

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # =========================
# # ImageNet mean/std
# # =========================
# mean = [0.485, 0.456, 0.406]
# std  = [0.229, 0.224, 0.225]

# train_transforms = transforms.Compose([
#     transforms.RandomResizedCrop(224, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
#     transforms.RandomHorizontalFlip(p=0.5),
#     transforms.RandomRotation(degrees=10),
#     transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.0),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=mean, std=std),
# ])

# val_transforms = transforms.Compose([
#     transforms.Resize(size=(256)),
#     transforms.CenterCrop(size=(224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=mean, std=std)
# ])

# gdrive_dataset_path = '/content/drive/MyDrive/치아질환분류데이터셋'

# train_dataset_v2 = datasets.ImageFolder(f"{gdrive_dataset_path}/train", transform=train_transforms)
# val_dataset_v2   = datasets.ImageFolder(f"{gdrive_dataset_path}/val",   transform=val_transforms)

# # =========================
# # ✅ WeightedRandomSampler (train만)
# # =========================
# labels = torch.tensor(train_dataset_v2.targets, dtype=torch.long)
# num_classes = len(train_dataset_v2.classes)

# class_count = torch.bincount(labels, minlength=num_classes).float().clamp_min(1.0)
# alpha = 1.0
# class_weight = (1.0 / class_count) ** alpha
# sample_weight = class_weight[labels]

# SEED = 42
# g = torch.Generator()
# g.manual_seed(SEED)

# train_sampler = WeightedRandomSampler(
#     weights=sample_weight,
#     num_samples=len(sample_weight),
#     replacement=True,
#     generator=g
# )

# print("Class names:", train_dataset_v2.classes)
# print("Train class_count:", class_count.tolist())
# print("Train class_weight(alpha={}):".format(alpha), class_weight.tolist())

# train_loader_v2 = DataLoader(
#     train_dataset_v2,
#     batch_size=32,
#     sampler=train_sampler,   # ✅ sampler 사용
#     shuffle=False,
#     num_workers=0,
#     pin_memory=True
# )

# val_loader_v2 = DataLoader(
#     val_dataset_v2,
#     batch_size=32,
#     shuffle=False,
#     num_workers=0,
#     pin_memory=True
# )

# # =========================
# # ✅ VGG19_BN + BN만 학습 + classifier만 학습 (네 방식 유지)
# # =========================
# vggnet_v2 = models.vgg19_bn(weights=VGG19_BN_Weights.DEFAULT)

# for param in vggnet_v2.parameters():
#     param.requires_grad = False

# for m in vggnet_v2.modules():
#     if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
#         if m.weight is not None:
#             m.weight.requires_grad = True
#         if m.bias is not None:
#             m.bias.requires_grad = True

# vggnet_v2.classifier[6] = nn.Linear(4096, num_classes)
# for param in vggnet_v2.classifier[6].parameters():
#     param.requires_grad = True

# vggnet_v2 = vggnet_v2.to(device)

# criterion_v2 = nn.CrossEntropyLoss()

# # =========================
# # ✅ 학습 실행
# # ✅ 이번 1차 실험: CutMix ON(0.5), Mixup OFF
# # =========================
# train_losses_v2, val_losses_v2, val_accuracies_v2, val_macro_f1s_v2, val_balanced_accuracies_v2, val_per_class_recalls_v2, \
# ema_gap_history_v2, start_epoch_v2, best_val_macro_f1_v2, best_val_balanced_accuracy_v2, best_class_names_v2, best_per_class_recall_v2, best_val_loss_v2 = \
# train_model_v2(
#     optimizer_name='SGD',
#     net=vggnet_v2,
#     train_loader=train_loader_v2,
#     val_loader=val_loader_v2,
#     criterion=criterion_v2,
#     num_epochs=100,
#     device=device,
#     ckpt_path="best_full_v2.pt",

#     # ✅ [핵심] CutMix ON(0.5), Mixup OFF
#     use_cutmix=False, #CUTMIX도움안되서 끔
#     cutmix_prob=0.5,
#     cutmix_alpha=1.0,
#     use_mixup=False,
#     mixup_alpha=0.2
# )

# print(f"\n최고 성능 (Early Stopping 기준 - Macro-F1):")
# print(f"  Best Validation Macro-F1: {best_val_macro_f1_v2:.4f}")
# print(f"  Best Validation Balanced Accuracy: {best_val_balanced_accuracy_v2*100:.2f}%")

# if best_per_class_recall_v2 is not None:
#     best_recall_str = " | ".join([f"{n}:{r:.3f}" for n, r in zip(best_class_names_v2, best_per_class_recall_v2)])
#     print(f"  Best Per-class Recall: {best_recall_str}")
# else:
#     print("  Best Per-class Recall: (없음) - ckpt 저장값 확인 필요")

## 클래스 조건부 blur / downsample 증강 실험
아래 코드는 당시 실험 흔적을 그대로 보존한 아카이브 블록입니다.

In [ ]:
# import torch
# import torch.nn as nn
# import torchvision.models as models
# from torchvision.models import VGG19_BN_Weights
# from torchvision import datasets, transforms
# from torch.utils.data import DataLoader, WeightedRandomSampler

# # ✅ [변경] PIL 기반 조건부 증강 구현을 위한 import
# from PIL import Image, ImageFilter
# import io
# import random

# # ✅ [중요] 1번 파일에서 train_model_v2를 import 해서 사용한다고 가정
# # from model_train_v2 import train_model_v2

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # =========================
# # ImageNet mean/std
# # =========================
# mean = [0.485, 0.456, 0.406]
# std  = [0.229, 0.224, 0.225]

# gdrive_dataset_path = '/content/drive/MyDrive/치아질환분류데이터셋'

# # ============================================================
# # ✅ [변경] 1) "라벨을 보고" 조건부 증강을 적용할 Dataset 래퍼
# # ============================================================
# class ClassConditionalImageFolder(torch.utils.data.Dataset):
#     """
#     ImageFolder를 감싸서:
#     - (PIL 단계) 조건부 증강(특정 클래스에만 적용)
#     - (PIL 단계) 공통 증강
#     - (Tensor 단계) ToTensor/Normalize
#     를 순서대로 적용한다.
#     """
#     def __init__(self, base_imagefolder: datasets.ImageFolder,
#                  conditional_pil_transform=None,
#                  common_pil_transform=None,
#                  tensor_transform=None):
#         self.base = base_imagefolder
#         self.conditional_pil_transform = conditional_pil_transform
#         self.common_pil_transform = common_pil_transform
#         self.tensor_transform = tensor_transform

#         # ImageFolder가 가진 메타를 그대로 노출
#         self.classes = base_imagefolder.classes
#         self.class_to_idx = base_imagefolder.class_to_idx
#         self.samples = base_imagefolder.samples
#         self.targets = base_imagefolder.targets

#     def __len__(self):
#         return len(self.base)

#     def __getitem__(self, idx):
#         path, target = self.samples[idx]

#         # ✅ [변경] ImageFolder 기본 loader 사용(PIL 이미지 로드)
#         img = self.base.loader(path)  # PIL Image

#         # ✅ [변경] 조건부(클래스별) PIL 증강
#         if self.conditional_pil_transform is not None:
#             img = self.conditional_pil_transform(img, target)

#         # ✅ [변경] 공통 PIL 증강
#         if self.common_pil_transform is not None:
#             img = self.common_pil_transform(img)

#         # ✅ [변경] 텐서 변환 + 정규화
#         if self.tensor_transform is not None:
#             img = self.tensor_transform(img)

#         return img, target


# # ============================================================
# # ✅ [변경] 2) ulcers/discoloration에만 적용할 "품질 열화" 증강
# #    - blur(선명도 낮추기)
# #    - downsample->upsample(해상도 열화)
# #    - (선택) JPEG 재압축(필요하면 ON)
# # ============================================================
# class QualityDegradeAug:
#     def __init__(self,
#                  target_class_indices,
#                  p_apply=0.7,
#                  p_blur=0.7,
#                  blur_radius=(1.0, 2.5),
#                  p_downsample=0.7,
#                  downsample_scale=(0.35, 0.7),
#                  p_jpeg=0.0,           # ✅ [변경] 기본 OFF (원하면 0.2~0.5로 올리기)
#                  jpeg_quality=(20, 50),
#                  seed=42):
#         self.target_set = set(target_class_indices)
#         self.p_apply = p_apply
#         self.p_blur = p_blur
#         self.blur_radius = blur_radius
#         self.p_downsample = p_downsample
#         self.downsample_scale = downsample_scale
#         self.p_jpeg = p_jpeg
#         self.jpeg_quality = jpeg_quality

#         self.rng = random.Random(seed)

#     def _maybe_blur(self, img: Image.Image) -> Image.Image:
#         if self.rng.random() < self.p_blur:
#             r = self.rng.uniform(*self.blur_radius)
#             img = img.filter(ImageFilter.GaussianBlur(radius=r))
#         return img

#     def _maybe_downsample(self, img: Image.Image) -> Image.Image:
#         if self.rng.random() < self.p_downsample:
#             w, h = img.size
#             s = self.rng.uniform(*self.downsample_scale)
#             nw, nh = max(32, int(w * s)), max(32, int(h * s))
#             img_small = img.resize((nw, nh), resample=Image.BILINEAR)
#             img = img_small.resize((w, h), resample=Image.BILINEAR)
#         return img

#     def _maybe_jpeg(self, img: Image.Image) -> Image.Image:
#         if self.p_jpeg > 0 and (self.rng.random() < self.p_jpeg):
#             q = self.rng.randint(*self.jpeg_quality)
#             buf = io.BytesIO()
#             img.save(buf, format="JPEG", quality=q, optimize=True)
#             buf.seek(0)
#             img = Image.open(buf).convert("RGB")
#         return img

#     def __call__(self, img: Image.Image, target: int) -> Image.Image:
#         # ✅ [변경] 특정 클래스에서만 적용
#         if target not in self.target_set:
#             return img

#         # ✅ [변경] 확률적으로만 적용(너무 과격해지는 걸 방지)
#         if self.rng.random() >= self.p_apply:
#             return img

#         # 순서: downsample(해상도 손실) -> blur -> (옵션) jpeg
#         img = self._maybe_downsample(img)
#         img = self._maybe_blur(img)
#         img = self._maybe_jpeg(img)
#         return img


# # ============================================================
# # ✅ [변경] 3) 공통 transform을 "PIL 단계"와 "Tensor 단계"로 분리
# # ============================================================
# common_pil_train = transforms.Compose([
#     transforms.RandomResizedCrop(224, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
#     transforms.RandomHorizontalFlip(p=0.5),
#     transforms.RandomRotation(degrees=10),
#     transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.0),
# ])

# tensor_norm = transforms.Compose([
#     transforms.ToTensor(),
#     transforms.Normalize(mean=mean, std=std),
# ])

# val_transforms = transforms.Compose([
#     transforms.Resize(size=(256)),
#     transforms.CenterCrop(size=(224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=mean, std=std)
# ])

# # ============================================================
# # ✅ [변경] 4) ImageFolder를 transform=None로 먼저 만들고,
# #            클래스 인덱스(ulcers/discoloration)를 얻은 뒤 래핑
# # ============================================================
# base_train = datasets.ImageFolder(f"{gdrive_dataset_path}/train", transform=None)
# base_val   = datasets.ImageFolder(f"{gdrive_dataset_path}/val",   transform=val_transforms)

# print("Class names:", base_train.classes)

# # ✅ [변경] target 클래스 인덱스 찾기
# target_names = ["ulcers", "discoloration"]
# missing = [n for n in target_names if n not in base_train.class_to_idx]
# if len(missing) > 0:
#     raise ValueError(f"train 폴더 클래스에 {missing} 가 없습니다. class_to_idx={base_train.class_to_idx}")

# target_class_indices = [base_train.class_to_idx[n] for n in target_names]
# print("Target class indices (conditional aug):", {n: base_train.class_to_idx[n] for n in target_names})

# # ✅ [변경] 조건부 품질 열화 증강 생성
# conditional_aug = QualityDegradeAug(
#     target_class_indices=target_class_indices,
#     p_apply=0.75,            # ✅ [튜닝] ulcers/discoloration에 적용 확률 (0.6~0.9)
#     p_blur=0.80,             # ✅ [튜닝] blur 적용 확률
#     blur_radius=(1.2, 3.0),  # ✅ [튜닝] blur 강도
#     p_downsample=0.80,       # ✅ [튜닝] downsample 적용 확률
#     downsample_scale=(0.35, 0.70),  # ✅ [튜닝] 해상도 열화 강도 (작을수록 더 심함)
#     p_jpeg=0.0,              # ✅ [튜닝] 필요하면 0.2로 켜보기
#     jpeg_quality=(20, 55),
#     seed=42
# )

# # ✅ [변경] 최종 train_dataset: 조건부증강 + 공통증강 + 정규화
# train_dataset_v2 = ClassConditionalImageFolder(
#     base_imagefolder=base_train,
#     conditional_pil_transform=conditional_aug,
#     common_pil_transform=common_pil_train,
#     tensor_transform=tensor_norm
# )

# val_dataset_v2 = base_val  # val은 기존대로 (평가 분포를 건드리지 않기 위해)

# # =========================
# # ✅ WeightedRandomSampler (train만)
# # =========================
# labels = torch.tensor(train_dataset_v2.targets, dtype=torch.long)
# num_classes = len(train_dataset_v2.classes)

# class_count = torch.bincount(labels, minlength=num_classes).float().clamp_min(1.0)
# alpha = 1.0
# class_weight = (1.0 / class_count) ** alpha
# sample_weight = class_weight[labels]

# SEED = 42
# g = torch.Generator()
# g.manual_seed(SEED)

# train_sampler = WeightedRandomSampler(
#     weights=sample_weight,
#     num_samples=len(sample_weight),
#     replacement=True,
#     generator=g
# )

# print("Train class_count:", class_count.tolist())
# print("Train class_weight(alpha={}):".format(alpha), class_weight.tolist())

# train_loader_v2 = DataLoader(
#     train_dataset_v2,
#     batch_size=32,
#     sampler=train_sampler,   # ✅ sampler 사용
#     shuffle=False,
#     num_workers=0,
#     pin_memory=True
# )

# val_loader_v2 = DataLoader(
#     val_dataset_v2,
#     batch_size=32,
#     shuffle=False,
#     num_workers=0,
#     pin_memory=True
# )

# # =========================
# # ✅ VGG19_BN + BN만 학습 + classifier만 학습 (네 방식 유지)
# # =========================
# vggnet_v2 = models.vgg19_bn(weights=VGG19_BN_Weights.DEFAULT)

# for param in vggnet_v2.parameters():
#     param.requires_grad = False

# for m in vggnet_v2.modules():
#     if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
#         if m.weight is not None:
#             m.weight.requires_grad = True
#         if m.bias is not None:
#             m.bias.requires_grad = True

# vggnet_v2.classifier[6] = nn.Linear(4096, num_classes)
# for param in vggnet_v2.classifier[6].parameters():
#     param.requires_grad = True

# vggnet_v2 = vggnet_v2.to(device)

# criterion_v2 = nn.CrossEntropyLoss()

# # =========================
# # ✅ 학습 실행
# # ✅ 이번 실험: 조건부 품질 열화 증강 ON (ulcers/discoloration)
# # =========================
# train_losses_v2, val_losses_v2, val_accuracies_v2, val_macro_f1s_v2, val_balanced_accuracies_v2, val_per_class_recalls_v2, \
# ema_gap_history_v2, start_epoch_v2, best_val_macro_f1_v2, best_val_balanced_accuracy_v2, best_class_names_v2, best_per_class_recall_v2, best_val_loss_v2 = \
# train_model_v2(
#     optimizer_name='SGD',
#     net=vggnet_v2,
#     train_loader=train_loader_v2,
#     val_loader=val_loader_v2,
#     criterion=criterion_v2,
#     num_epochs=100,
#     device=device,
#     ckpt_path="best_full_v2.pt",

#     # ✅ [유지] CutMix OFF, Mixup OFF
#     use_cutmix=False,
#     cutmix_prob=0.5,
#     cutmix_alpha=1.0,
#     use_mixup=False,
#     mixup_alpha=0.2
# )

# print(f"\n최고 성능 (Early Stopping 기준 - Macro-F1):")
# print(f"  Best Validation Macro-F1: {best_val_macro_f1_v2:.4f}")
# print(f"  Best Validation Balanced Accuracy: {best_val_balanced_accuracy_v2*100:.2f}%")

# if best_per_class_recall_v2 is not None:
#     best_recall_str = " | ".join([f"{n}:{r:.3f}" for n, r in zip(best_class_names_v2, best_per_class_recall_v2)])
#     print(f"  Best Per-class Recall: {best_recall_str}")
# else:
    # print("  Best Per-class Recall: (없음) - ckpt 저장값 확인 필요")

## 원본 + 재분할 연속 실행 통합 러너(이전 버전)
아래 코드는 당시 실험 흔적을 그대로 보존한 아카이브 블록입니다.

In [ ]:
# import os
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import torchvision.models as models
# from torchvision.models import VGG19_BN_Weights, ResNet50_Weights, DenseNet121_Weights
# from torchvision import datasets, transforms
# from torch.utils.data import DataLoader, WeightedRandomSampler

# import numpy as np
# import matplotlib.pyplot as plt

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # ============================================================
# # ✅ Drive 저장 루트(여기 아래에 orig / resplit 폴더로 분리 저장됨)
# # ============================================================
# SAVE_ROOT = "/content/drive/MyDrive/model_compare_bestft_fixedclass_runs"
# os.makedirs(SAVE_ROOT, exist_ok=True)
# print("✅ SAVE_ROOT =", SAVE_ROOT)

# # ============================================================
# # ✅ 2번 연속 실행할 데이터셋 루트 목록
# # ============================================================
# RUNS = [
#     ("orig", "/content/drive/MyDrive/치아질환분류데이터셋"),
#     ("resplit", "/content/drive/MyDrive/치아질환분류데이터셋_resplit_classwise_v1"),
# ]

# # ============================================================
# # ✅ 클래스 순서 고정
# # ============================================================
# FIXED_CLASSES = ["calculus", "caries", "discoloration", "hypodontia", "ulcers"]

# class RemapTargetsDataset(torch.utils.data.Dataset):
#     def __init__(self, base_ds, fixed_classes):
#         self.base = base_ds
#         self.fixed_classes = list(fixed_classes)
#         self.fixed_class_to_idx = {c:i for i,c in enumerate(self.fixed_classes)}

#         ds_set = set(base_ds.classes)
#         fixed_set = set(self.fixed_classes)
#         if ds_set != fixed_set:
#             missing = sorted(list(fixed_set - ds_set))
#             extra = sorted(list(ds_set - fixed_set))
#             raise ValueError(f"[class mismatch] missing={missing}, extra={extra}. base_ds.classes={base_ds.classes}")

#         inv_old = {old_idx: cls for cls, old_idx in base_ds.class_to_idx.items()}
#         self.old_to_fixed = {old_idx: self.fixed_class_to_idx[inv_old[old_idx]] for old_idx in inv_old}

#         self.classes = self.fixed_classes
#         self.class_to_idx = self.fixed_class_to_idx
#         self.targets = [self.old_to_fixed[int(t)] for t in base_ds.targets]

#     def __len__(self):
#         return len(self.base)

#     def __getitem__(self, idx):
#         x, y_old = self.base[idx]
#         y = self.old_to_fixed[int(y_old)]
#         return x, y

# # ============================================================
# # transforms (공통)
# # ============================================================
# mean = [0.485, 0.456, 0.406]
# std  = [0.229, 0.224, 0.225]

# train_transforms = transforms.Compose([
#     transforms.RandomResizedCrop(224, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
#     transforms.RandomHorizontalFlip(p=0.5),
#     transforms.RandomRotation(degrees=10),
#     transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.0),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=mean, std=std),
# ])

# eval_transforms = transforms.Compose([
#     transforms.Resize(size=(256)),
#     transforms.CenterCrop(size=(224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=mean, std=std)
# ])

# # ============================================================
# # 모델별 fine-tuning 정책 (공통 함수)
# # ============================================================
# def freeze_all(net):
#     for p in net.parameters():
#         p.requires_grad = False

# def enable_bn_affine(net):
#     for m in net.modules():
#         if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
#             if m.weight is not None: m.weight.requires_grad = True
#             if m.bias is not None:   m.bias.requires_grad = True

# def build_vgg19_bn(num_classes):
#     net = models.vgg19_bn(weights=VGG19_BN_Weights.DEFAULT)
#     freeze_all(net)
#     enable_bn_affine(net)
#     net.classifier[6] = nn.Linear(4096, num_classes)
#     for p in net.classifier[6].parameters():
#         p.requires_grad = True
#     return net

# def build_resnet50_bestft(num_classes):
#     net = models.resnet50(weights=ResNet50_Weights.DEFAULT)
#     freeze_all(net)
#     for p in net.layer4.parameters():
#         p.requires_grad = True
#     enable_bn_affine(net)
#     net.fc = nn.Linear(net.fc.in_features, num_classes)
#     for p in net.fc.parameters():
#         p.requires_grad = True
#     return net

# def build_densenet121_bestft(num_classes):
#     net = models.densenet121(weights=DenseNet121_Weights.DEFAULT)
#     freeze_all(net)
#     for p in net.features.denseblock4.parameters():
#         p.requires_grad = True
#     if hasattr(net.features, "norm5"):
#         for p in net.features.norm5.parameters():
#             p.requires_grad = True
#     enable_bn_affine(net)
#     net.classifier = nn.Linear(net.classifier.in_features, num_classes)
#     for p in net.classifier.parameters():
#         p.requires_grad = True
#     return net

# def make_sgd_param_groups(net, head_keys, backbone_keys,
#                           lr_head=1e-3, lr_backbone=1e-4, lr_bn=3e-4,
#                           weight_decay=5e-4, momentum=0.9):
#     head_params, backbone_params, bn_params = [], [], []
#     for name, p in net.named_parameters():
#         if not p.requires_grad:
#             continue
#         if "bn" in name.lower() or "batchnorm" in name.lower():
#             bn_params.append(p)
#             continue
#         if any(k in name for k in head_keys):
#             head_params.append(p)
#         elif any(k in name for k in backbone_keys):
#             backbone_params.append(p)
#         else:
#             backbone_params.append(p)

#     param_groups = []
#     if head_params:
#         param_groups.append({"params": head_params, "lr": lr_head, "weight_decay": weight_decay, "momentum": momentum})
#     if backbone_params:
#         param_groups.append({"params": backbone_params, "lr": lr_backbone, "weight_decay": weight_decay, "momentum": momentum})
#     if bn_params:
#         param_groups.append({"params": bn_params, "lr": lr_bn, "weight_decay": 0.0, "momentum": momentum})

#     return optim.SGD(param_groups, lr=lr_head, momentum=momentum, weight_decay=weight_decay)

# # ============================================================
# # TEST 평가
# # ============================================================
# @torch.no_grad()
# def evaluate_on_loader(net, loader, criterion, num_classes, device):
#     net.eval()
#     cm = torch.zeros((num_classes, num_classes), dtype=torch.int64)
#     total_loss, total_n = 0.0, 0

#     for x, y in loader:
#         x = x.to(device); y = y.to(device)
#         logits = net(x)
#         loss = criterion(logits, y)
#         bs = y.size(0)
#         total_loss += float(loss.item()) * bs
#         total_n += bs

#         pred = logits.argmax(dim=1)
#         for t, p in zip(y.view(-1), pred.view(-1)):
#             cm[t.long(), p.long()] += 1

#     cm_f = cm.float()
#     tp = torch.diag(cm_f)
#     fn = cm_f.sum(dim=1) - tp
#     fp = cm_f.sum(dim=0) - tp

#     eps = 1e-12
#     recall = tp / (tp + fn + eps)
#     precision = tp / (tp + fp + eps)
#     f1 = 2 * precision * recall / (precision + recall + eps)

#     macro_f1 = float(f1.mean().item())
#     balanced_acc = float(recall.mean().item())
#     acc = float(tp.sum().item() / max(cm_f.sum().item(), 1.0))
#     avg_loss = total_loss / max(total_n, 1)

#     return cm.cpu().numpy(), recall.cpu().numpy(), acc, macro_f1, balanced_acc, avg_loss

# def normalize_rows(cm):
#     row_sum = cm.sum(axis=1, keepdims=True)
#     row_sum[row_sum == 0] = 1
#     return cm / row_sum

# criterion = nn.CrossEntropyLoss()

# MODELS = [
#     ("vgg19_bn", build_vgg19_bn),
#     ("resnet50", build_resnet50_bestft),
#     ("densenet121", build_densenet121_bestft),
# ]

# # ============================================================
# # ✅ RUN 루프: orig 먼저, 그 다음 resplit
# # ============================================================
# for RUN_TAG, DATA_ROOT in RUNS:
#     print("\n" + "#"*90)
#     print(f"### RUN = {RUN_TAG} | DATA_ROOT = {DATA_ROOT}")
#     print("#"*90)

#     run_out = os.path.join(SAVE_ROOT, RUN_TAG)
#     os.makedirs(run_out, exist_ok=True)
#     print("✅ run_out =", run_out)

#     # dataset 로드
#     train_ds_raw = datasets.ImageFolder(f"{DATA_ROOT}/train", transform=train_transforms)
#     val_ds_raw   = datasets.ImageFolder(f"{DATA_ROOT}/val",   transform=eval_transforms)
#     test_ds_raw  = datasets.ImageFolder(f"{DATA_ROOT}/test",  transform=eval_transforms)

#     train_ds = RemapTargetsDataset(train_ds_raw, FIXED_CLASSES)
#     val_ds   = RemapTargetsDataset(val_ds_raw, FIXED_CLASSES)
#     test_ds  = RemapTargetsDataset(test_ds_raw, FIXED_CLASSES)

#     class_names = train_ds.classes
#     num_classes = len(class_names)

#     print("FIXED class order:", class_names)
#     print("fixed class_to_idx:", train_ds.class_to_idx)

#     # sampler
#     labels = torch.tensor(train_ds.targets, dtype=torch.long)
#     class_count = torch.bincount(labels, minlength=num_classes).float().clamp_min(1.0)
#     class_weight = (1.0 / class_count) ** 1.0
#     sample_weight = class_weight[labels]

#     g = torch.Generator(); g.manual_seed(42)
#     train_sampler = WeightedRandomSampler(sample_weight, num_samples=len(sample_weight), replacement=True, generator=g)

#     train_loader = DataLoader(train_ds, batch_size=32, sampler=train_sampler, shuffle=False, num_workers=0, pin_memory=True)
#     val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0, pin_memory=True)
#     test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

#     results = []

#     for name, builder in MODELS:
#         print("\n" + "="*80)
#         print(f"Training: {name} ({RUN_TAG})")
#         print("="*80)

#         net = builder(num_classes).to(device)

#         if name == "vgg19_bn":
#             optimizer = None
#         elif name == "resnet50":
#             optimizer = make_sgd_param_groups(net, head_keys=["fc."], backbone_keys=["layer4."],
#                                               lr_head=1e-3, lr_backbone=1e-4, lr_bn=3e-4)
#         else:
#             optimizer = make_sgd_param_groups(net, head_keys=["classifier."],
#                                               backbone_keys=["features.denseblock4.", "features.norm5."],
#                                               lr_head=1e-3, lr_backbone=1e-4, lr_bn=3e-4)

#         ckpt_path = os.path.join(run_out, f"best_{name}_{RUN_TAG}.pt")

#         # ✅ train_model_v2는 셀1에 정의되어 있어야 함
#         train_model_v2(
#             optimizer_name="SGD",
#             net=net,
#             train_loader=train_loader,
#             val_loader=val_loader,
#             criterion=criterion,
#             num_epochs=100,
#             device=device,
#             ckpt_path=ckpt_path,
#             use_cutmix=False,
#             use_mixup=False,
#             optimizer=optimizer,
#             default_lr=3e-5
#         )

#         ckpt = torch.load(ckpt_path, map_location=device)
#         if "model_state_dict" in ckpt:
#             net.load_state_dict(ckpt["model_state_dict"], strict=True)
#         else:
#             net.load_state_dict(ckpt["model"], strict=True)

#         cm, recall, acc, macro_f1, bal_acc, loss = evaluate_on_loader(net, test_loader, criterion, num_classes, device)
#         cm_norm = normalize_rows(cm)

#         recall_str = " | ".join([f"{cls}:{recall[i]:.3f}" for i, cls in enumerate(class_names)])
#         print(f"[{name}/{RUN_TAG}] TEST loss={loss:.4f} | acc={acc*100:.2f}% | macroF1={macro_f1:.4f} | bal_acc={bal_acc*100:.2f}%")
#         print(f"[{name}/{RUN_TAG}] TEST Per-class Recall: {recall_str}")

#         results.append({
#             "model": name,
#             "cm_norm": cm_norm,
#             "recall": recall,
#             "acc": acc,
#             "macro_f1": macro_f1,
#             "bal_acc": bal_acc,
#             "loss": loss
#         })

#     # ✅ 리포트 저장(덮어쓰기 방지: RUN_TAG 포함)
#     table_cols = ["Class"] + [r["model"] for r in results]
#     table_data = []
#     for i, cls in enumerate(class_names):
#         table_data.append([cls] + [f"{r['recall'][i]:.3f}" for r in results])

#     overall_cols = ["Metric"] + [r["model"] for r in results]
#     overall_data = [
#         ["Test Loss"] + [f"{r['loss']:.4f}" for r in results],
#         ["Test Acc"] + [f"{r['acc']*100:.2f}%" for r in results],
#         ["Test Macro-F1"] + [f"{r['macro_f1']:.4f}" for r in results],
#         ["Test Balanced Acc"] + [f"{r['bal_acc']*100:.2f}%" for r in results],
#     ]

#     fig = plt.figure(figsize=(18, 11))
#     for idx, r in enumerate(results):
#         ax = fig.add_subplot(2, 3, idx+1)
#         im = ax.imshow(r["cm_norm"], aspect="auto")
#         ax.set_title(f"{r['model']} ({RUN_TAG}) normalized CM")
#         ax.set_xlabel("Pred"); ax.set_ylabel("True")
#         ax.set_xticks(range(num_classes)); ax.set_yticks(range(num_classes))
#         ax.set_xticklabels(class_names, rotation=45, ha="right")
#         ax.set_yticklabels(class_names)

#         for i in range(num_classes):
#             for j in range(num_classes):
#                 ax.text(j, i, f"{r['cm_norm'][i,j]:.2f}", ha="center", va="center", fontsize=7)

#         fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

#     ax_tbl = fig.add_subplot(2, 1, 2)
#     ax_tbl.axis("off")

#     tbl1 = ax_tbl.table(cellText=table_data, colLabels=table_cols, loc="upper center")
#     tbl1.auto_set_font_size(False); tbl1.set_fontsize(10); tbl1.scale(1.0, 1.25)

#     tbl2 = ax_tbl.table(cellText=overall_data, colLabels=overall_cols, loc="lower center")
#     tbl2.auto_set_font_size(False); tbl2.set_fontsize(11); tbl2.scale(1.0, 1.35)

#     ax_tbl.set_title(f"TEST Report ({RUN_TAG}): Per-class Recall + Overall Metrics", pad=10)

#     plt.tight_layout()
#     pdf_path = os.path.join(run_out, f"report_onepage_{RUN_TAG}.pdf")
#     png_path = os.path.join(run_out, f"report_onepage_{RUN_TAG}.png")
#     plt.savefig(pdf_path)
#     plt.savefig(png_path, dpi=220)
#     plt.close()

#     print("\n✅ Saved report to Drive:")
#     print(" -", pdf_path)
#     print(" -", png_path)
#     print("✅ Saved ckpts to Drive inside:", run_out)